# Pipeline de alineación — tejido renal (VALIS + tiatoolbox)

Separa las dos muestras del TIF de entrada, las alinea con VALIS o tiatoolbox (según `MOTOR_REGISTRO`) y extrae tiles pareados para su posterior análisis. El Paso 9 compara ambos motores de registro y el Paso 10 exporta los parámetros utilizados en la corrida.


In [ ]:
RUTA_TIF_ORIGINAL  = "C:/investigacion/107514-40x.tif"
CARPETA_RECORTES   = "C:/investigacion/paper/muestras/107514-40x_recortes"
CARPETA_RESULTADOS = "C:/investigacion/paper/muestras/20107514-40x_resultados_tiatoolbox"
CARPETA_TILES_BASE = "C:/investigacion/paper/muestras/107514-40x_tiles_tiatoolbox"

MOTOR_REGISTRO = "valis"
assert MOTOR_REGISTRO in ("valis", "tiatoolbox"), "MOTOR_REGISTRO debe ser 'valis' o 'tiatoolbox'"

TILE_SIZE   = 512
STRIDE      = 256
NIVEL_TILES = 0

UMBRAL_TEJIDO      = 0.4
MIN_INFORMATIVIDAD = 15.0

SEMILLA = 42

SEPARAR_LADO_MINIATURA_PX     = 2000
SEPARAR_MARGEN_PCT            = 0.02
SEPARAR_TAMANIO_MIN_OBJETO_PX = 500

N_TILES_VERIFICACION   = 8
MARGEN_BUSQUEDA_PX     = 200
UMBRAL_OFFSET_PX       = 5
UMBRAL_CONSISTENCIA_PX = 25

CORRER_MICRO_REGISTRO             = False
VALIS_MAX_PROCESSED_IMG_DIM_PX    = 1024
VALIS_MAX_NON_RIGID_DIM_PX        = 2048
VALIS_MICRO_MAX_NON_RIGID_DIM_PX  = 4000

VALIS_USAR_FEATURES_DEEP         = True
VALIS_N_FEATURES                 = 2000
VALIS_USAR_MICRO_RIGID_REGISTRAR = True
VALIS_CHECK_FOR_REFLECTIONS      = False

VALIS_NON_RIGID_REGISTRAR = "simpleelastix"

TIATOOLBOX_DFBR_MAX_DIM_PX          = 2000
TIATOOLBOX_DFBR_DEVICE              = "cuda"
TIATOOLBOX_REFINAR_BSPLINE_POR_TILE = True
TIATOOLBOX_BSPLINE_GRID_SPACE_UM    = 100.0
TIATOOLBOX_BSPLINE_SAMPLING_PERCENT = 0.1

UMBRAL_STD_TEXTURA           = 5.0
UMBRAL_RESPONSE_FALLA        = 0.01
UMBRAL_SHIFT_FALLA_RELATIVO  = 20
PERCENTIL_FALLA_MEDICION     = 0.999

SHIFT_MAX_UM_CONTROL_VISUAL = 60
RESPONSE_MIN_CONTROL_VISUAL = 0.03

import os
import random

random.seed(SEMILLA)
try:
    import numpy as _np
    _np.random.seed(SEMILLA)
except ImportError:
    pass

CARPETA_TILES = f"{CARPETA_TILES_BASE}_tile{TILE_SIZE}_nivel{NIVEL_TILES}_{MOTOR_REGISTRO}"
os.makedirs(CARPETA_TILES, exist_ok=True)


---
## Paso 0 — Entorno


In [ ]:
import os
import sys

print(f"Python {sys.version.split()[0]}")

dependencias = [
    ("numpy", "numpy"), ("scikit-image", "skimage"), ("tiatoolbox", "tiatoolbox"),
    ("pyvips", "pyvips"), ("valis", "valis"), ("matplotlib", "matplotlib"),
    ("opencv", "cv2"), ("pandas", "pandas"), ("torch", "torch"),
]
for nombre, modulo in dependencias:
    try:
        mod = __import__(modulo)
        print(f"  {nombre:<14} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {nombre:<14} no instalado")

try:
    import torch
    print(f"GPU (torch.cuda.is_available): {torch.cuda.is_available()}")
except ImportError:
    pass

try:
    from valis import feature_detectors, feature_matcher
    tiene_disk = hasattr(feature_detectors, "DiskFD")
    tiene_lightglue = hasattr(feature_matcher, "LightGlueMatcher")
    print(f"VALIS DISK+LightGlue disponible: {tiene_disk and tiene_lightglue}")
except ImportError:
    pass

try:
    from valis.micro_rigid_registrar import MicroRigidRegistrar  # noqa: F401
except ImportError:
    pass

try:
    from tiatoolbox.tools.registration.wsi_registration import (  # noqa: F401
        DFBRegister, AffineWSITransformer, estimate_bspline_transform,
        apply_bspline_transform, match_histograms,
    )
except ImportError as _e:
    print(f"Módulo de registro de tiatoolbox no disponible: {_e}")

_ruta_m1 = os.path.join(CARPETA_RECORTES, "muestra_1.tif")
_ruta_m2 = os.path.join(CARPETA_RECORTES, "muestra_2.tif")
_ruta_offsets = os.path.join(CARPETA_RECORTES, "offsets_recorte.json")
_muestras_listas = all(os.path.exists(p) for p in (_ruta_m1, _ruta_m2, _ruta_offsets))
print(f"Muestras separadas disponibles: {_muestras_listas}")
print(f"Motor de registro configurado: {MOTOR_REGISTRO}")


---
## Paso 1 — Separación de las dos muestras del TIF

Utiliza el mismo método (Otsu, determinista) que `separar_muestras.py`, de modo que el resultado sea reproducible independientemente de quién ejecute el pipeline. Si `muestra_1.tif`, `muestra_2.tif` y `offsets_recorte.json` ya existen en `CARPETA_RECORTES`, se cargan directamente; en caso contrario, se generan en esta celda.


In [ ]:
import os
import json
import numpy as np
import pyvips
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from skimage.morphology import closing, disk, remove_small_objects, remove_small_holes
from skimage.measure import label, regionprops
from tiatoolbox.wsicore.wsireader import WSIReader

os.makedirs(CARPETA_RECORTES, exist_ok=True)
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

ruta_m1 = os.path.join(CARPETA_RECORTES, "muestra_1.tif")
ruta_m2 = os.path.join(CARPETA_RECORTES, "muestra_2.tif")
ruta_offsets = os.path.join(CARPETA_RECORTES, "offsets_recorte.json")

lector_original = WSIReader.open(RUTA_TIF_ORIGINAL)
mpp_base = lector_original.info.mpp[0]

if os.path.exists(ruta_m1) and os.path.exists(ruta_m2) and os.path.exists(ruta_offsets):
    with open(ruta_offsets) as f:
        offsets_recorte = json.load(f)
else:

    def _miniatura_numpy(img_vips, lado_max):
        lado_mayor = max(img_vips.width, img_vips.height)
        escala_reduccion = lado_max / lado_mayor
        mini_vips = img_vips.resize(escala_reduccion)
        if mini_vips.bands > 1:
            mini_vips = mini_vips.colourspace("b-w")
        buf = mini_vips.write_to_memory()
        mini_np = np.ndarray(buffer=buf, dtype=np.uint8, shape=[mini_vips.height, mini_vips.width])
        return mini_np, 1.0 / escala_reduccion

    def _detectar_2_regiones(mini_np):
        umbral = threshold_otsu(mini_np)
        mascara = mini_np < umbral
        mascara = remove_small_holes(mascara, area_threshold=SEPARAR_TAMANIO_MIN_OBJETO_PX)
        mascara = remove_small_objects(mascara, min_size=SEPARAR_TAMANIO_MIN_OBJETO_PX)
        mascara = closing(mascara, disk(3))
        etiquetas = label(mascara)
        regiones = sorted(regionprops(etiquetas), key=lambda r: r.area, reverse=True)
        if len(regiones) < 2:
            raise RuntimeError(
                f"Se detectaron {len(regiones)} región(es), se necesitan 2. "
                "Probar con un SEPARAR_TAMANIO_MIN_OBJETO_PX menor."
            )
        dos = regiones[:2]
        dos.sort(key=lambda r: r.centroid[1])
        return dos

    def _recortar_y_guardar(img_vips, region, escala_a_full, indice):
        fila_min, col_min, fila_max, col_max = region.bbox
        x = int(col_min * escala_a_full)
        y = int(fila_min * escala_a_full)
        ancho = int((col_max - col_min) * escala_a_full)
        alto = int((fila_max - fila_min) * escala_a_full)

        margen_x = int(ancho * SEPARAR_MARGEN_PCT)
        margen_y = int(alto * SEPARAR_MARGEN_PCT)
        x = max(0, x - margen_x)
        y = max(0, y - margen_y)
        ancho = min(img_vips.width - x, ancho + 2 * margen_x)
        alto = min(img_vips.height - y, alto + 2 * margen_y)

        recorte = img_vips.crop(x, y, ancho, alto)

        # La máscara binaria de la región (a resolución de miniatura) se
        # reescala y se aplica sobre el recorte para blanquear todo lo que
        # no pertenezca a esta muestra dentro del bounding box, evitando
        # que formas irregulares incluyan fondo o tejido de la otra muestra.
        mascara_region = region.image
        margen_mini_y = int(mascara_region.shape[0] * SEPARAR_MARGEN_PCT) + 1
        margen_mini_x = int(mascara_region.shape[1] * SEPARAR_MARGEN_PCT) + 1
        mascara_region = np.pad(
            mascara_region,
            ((margen_mini_y, margen_mini_y), (margen_mini_x, margen_mini_x)),
            mode="constant", constant_values=False
        )
        mascara_chica_vips = pyvips.Image.new_from_array(
            (mascara_region.astype(np.uint8)) * 255
        )
        mascara_grande = mascara_chica_vips.resize(
            ancho / mascara_chica_vips.width,
            vscale=alto / mascara_chica_vips.height,
            kernel="linear"
        ).crop(0, 0, ancho, alto)
        recorte = mascara_grande.ifthenelse(recorte, [255, 255, 255])

        ruta_salida = os.path.join(CARPETA_RECORTES, f"muestra_{indice}.tif")
        recorte.tiffsave(ruta_salida, tile=True, pyramid=True, bigtiff=True, compression="lzw")
        return {"x": x, "y": y, "ancho": ancho, "alto": alto}

    img_vips = pyvips.Image.new_from_file(RUTA_TIF_ORIGINAL, access="sequential")
    mini_np, escala_a_full = _miniatura_numpy(img_vips, SEPARAR_LADO_MINIATURA_PX)
    regiones = _detectar_2_regiones(mini_np)

    offsets_recorte = {}
    for i, region in enumerate(regiones, start=1):
        offsets_recorte[f"muestra_{i}"] = _recortar_y_guardar(img_vips, region, escala_a_full, i)

    with open(ruta_offsets, "w") as f:
        json.dump(offsets_recorte, f, indent=2)

ruta_metadata_pipeline = os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")
with open(ruta_metadata_pipeline, "w") as f:
    json.dump({
        "mpp_base": float(mpp_base),
        "offsets_recorte": offsets_recorte,
        "origen_recorte": "separar_muestras (Otsu, determinista)",
    }, f, indent=2)

print(f"mpp_base: {mpp_base:.4f} µm/px")
print(f"muestra_1: {offsets_recorte['muestra_1']}")
print(f"muestra_2: {offsets_recorte['muestra_2']}")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, ruta, nombre in zip(axes, (ruta_m1, ruta_m2), ("muestra_1", "muestra_2")):
    thumb = pyvips.Image.thumbnail(ruta, 800)
    buf = thumb.write_to_memory()
    arr = np.ndarray(buffer=buf, dtype=np.uint8, shape=[thumb.height, thumb.width, thumb.bands])
    ax.imshow(arr[..., :3])
    ax.set_title(nombre)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "muestras_separadas.png"), dpi=150, bbox_inches="tight")
plt.show()


---
## Paso 1.5 — Corrección de offset vs. GrandQC


In [ ]:
import re
import cv2
import random
import pandas as pd

if "BASE_GQC" not in globals():
    BASE_GQC = "C:/investigacion/datasetNefroTileCALIDAD"
if "T1_GQC" not in globals():
    T1_GQC = os.path.join(BASE_GQC, "ecc", "tejido1")
if "T2_GQC" not in globals():
    T2_GQC = os.path.join(BASE_GQC, "ecc", "tejido2")
TS_GQC_NOMINAL = 512

_PAT_GQC = re.compile(r"^tile_\d+_r(?P<y1>\d+)_c(?P<x1>\d+)\.\w+$", re.IGNORECASE)


def _listar_tiles_gqc(carpeta):
    if not os.path.isdir(carpeta):
        return []
    filas = []
    for nombre in os.listdir(carpeta):
        m = _PAT_GQC.match(nombre)
        if m:
            filas.append({"nombre": nombre, "x1_orig": int(m.group("x1")), "y1_orig": int(m.group("y1"))})
    return filas


def _vips_a_numpy(img_vips):
    buf = img_vips.write_to_memory()
    return np.ndarray(buffer=buf, dtype=np.uint8,
                       shape=[img_vips.height, img_vips.width, img_vips.bands])


def _medir_offset_residual(carpeta_gqc, ruta_recorte, offset_x, offset_y, nombre_muestra):
    tiles_gqc = _listar_tiles_gqc(carpeta_gqc)
    if not tiles_gqc or not os.path.exists(ruta_recorte):
        return None

    ancho_recorte = offsets_recorte[nombre_muestra]["ancho"]
    alto_recorte  = offsets_recorte[nombre_muestra]["alto"]

    candidatos = [
        t for t in tiles_gqc
        if offset_x - MARGEN_BUSQUEDA_PX <= t["x1_orig"] <= offset_x + ancho_recorte - TS_GQC_NOMINAL + MARGEN_BUSQUEDA_PX
        and offset_y - MARGEN_BUSQUEDA_PX <= t["y1_orig"] <= offset_y + alto_recorte - TS_GQC_NOMINAL + MARGEN_BUSQUEDA_PX
    ]
    if not candidatos:
        return None

    random.seed(SEMILLA)
    muestreados = random.sample(candidatos, min(N_TILES_VERIFICACION, len(candidatos)))

    img_recorte = pyvips.Image.new_from_file(ruta_recorte)
    ancho_img, alto_img = img_recorte.width, img_recorte.height

    residuos = []
    for t in muestreados:
        tile_gqc = cv2.imread(os.path.join(carpeta_gqc, t["nombre"]))
        if tile_gqc is None:
            continue
        tile_gqc_gray = cv2.cvtColor(tile_gqc, cv2.COLOR_BGR2GRAY)
        th, tw = tile_gqc_gray.shape[:2]

        lx = t["x1_orig"] - offset_x
        ly = t["y1_orig"] - offset_y

        bx0 = int(max(0, lx - MARGEN_BUSQUEDA_PX))
        by0 = int(max(0, ly - MARGEN_BUSQUEDA_PX))
        bx1 = int(min(ancho_img, lx + tw + MARGEN_BUSQUEDA_PX))
        by1 = int(min(alto_img,  ly + th + MARGEN_BUSQUEDA_PX))
        if bx1 - bx0 < tw or by1 - by0 < th:
            continue

        parche_grande = _vips_a_numpy(img_recorte.crop(bx0, by0, bx1 - bx0, by1 - by0))
        if parche_grande.shape[2] >= 3:
            parche_grande_gray = cv2.cvtColor(parche_grande[:, :, :3], cv2.COLOR_RGB2GRAY)
        else:
            parche_grande_gray = parche_grande[:, :, 0]

        resultado = cv2.matchTemplate(parche_grande_gray, tile_gqc_gray, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(resultado)

        naive_loc = (int(lx - bx0), int(ly - by0))
        naive_val = float(resultado[naive_loc[1], naive_loc[0]]) \
            if 0 <= naive_loc[1] < resultado.shape[0] and 0 <= naive_loc[0] < resultado.shape[1] else float("nan")

        dx = max_loc[0] - naive_loc[0]
        dy = max_loc[1] - naive_loc[1]
        residuos.append({"tile": t["nombre"], "dx": dx, "dy": dy,
                          "corr_naive": naive_val, "corr_mejor": max_val})

    if not residuos:
        return None

    df_res = pd.DataFrame(residuos)

    dx_mediana, dy_mediana = df_res["dx"].median(), df_res["dy"].median()
    dx_std, dy_std = df_res["dx"].std(), df_res["dy"].std()
    corr_naive_media = df_res["corr_naive"].mean()
    corr_mejor_media = df_res["corr_mejor"].mean()

    print(f"{nombre_muestra}: {len(df_res)} tiles | corr naive={corr_naive_media:.2f} "
          f"mejor={corr_mejor_media:.2f} | offset residual dx={dx_mediana:.0f}px (std {dx_std:.1f}), "
          f"dy={dy_mediana:.0f}px (std {dy_std:.1f})")

    if max(abs(dx_mediana), abs(dy_mediana)) < UMBRAL_OFFSET_PX:
        return {"dx": 0.0, "dy": 0.0, "confiable": True,
                "corr_naive": corr_naive_media, "corr_mejor": corr_mejor_media}

    confiable = not (dx_std > UMBRAL_CONSISTENCIA_PX or dy_std > UMBRAL_CONSISTENCIA_PX)
    return {"dx": float(dx_mediana), "dy": float(dy_mediana), "confiable": confiable,
            "corr_naive": corr_naive_media, "corr_mejor": corr_mejor_media}


if "offsets_recorte" not in globals():
    raise RuntimeError("Corré primero el Paso 1 (celda de recorte).")

correcciones = {}
for nombre_muestra, carpeta_gqc, ruta_recorte in [
    ("muestra_1", T1_GQC, os.path.join(CARPETA_RECORTES, "muestra_1.tif")),
    ("muestra_2", T2_GQC, os.path.join(CARPETA_RECORTES, "muestra_2.tif")),
]:
    off = offsets_recorte[nombre_muestra]
    resultado = _medir_offset_residual(carpeta_gqc, ruta_recorte, off["x"], off["y"], nombre_muestra)
    if resultado is not None:
        correcciones[nombre_muestra] = resultado

if correcciones:
    with open(ruta_metadata_pipeline, "r") as _f:
        _meta = json.load(_f)
    _meta["correccion_offset"] = {
        k: {"dx": v["dx"], "dy": v["dy"], "confiable": v["confiable"]}
        for k, v in correcciones.items()
    }

    ruta_csv_tiles_actual = os.path.join(CARPETA_TILES, "indice_tiles.csv")
    if os.path.exists(ruta_csv_tiles_actual):
        corr_m1 = correcciones.get("muestra_1")
        if corr_m1 is not None and corr_m1["confiable"] and \
           (abs(corr_m1["dx"]) >= UMBRAL_OFFSET_PX or abs(corr_m1["dy"]) >= UMBRAL_OFFSET_PX):
            _df_tiles = pd.read_csv(ruta_csv_tiles_actual)
            _df_tiles["x1_orig"] = _df_tiles["x1_orig"] + corr_m1["dx"]
            _df_tiles["y1_orig"] = _df_tiles["y1_orig"] + corr_m1["dy"]
            _df_tiles.to_csv(ruta_csv_tiles_actual, index=False)
    else:
        for nombre_muestra, corr in correcciones.items():
            if corr["confiable"] and (abs(corr["dx"]) >= UMBRAL_OFFSET_PX or abs(corr["dy"]) >= UMBRAL_OFFSET_PX):
                offsets_recorte[nombre_muestra]["x"] += int(round(corr["dx"]))
                offsets_recorte[nombre_muestra]["y"] += int(round(corr["dy"]))
        _meta["offsets_recorte"] = offsets_recorte

    with open(ruta_metadata_pipeline, "w") as _f:
        json.dump(_meta, _f, indent=2)


---
## Paso 2 — Registro (`MOTOR_REGISTRO`)

Ejecutar únicamente la celda correspondiente al motor configurado en `MOTOR_REGISTRO`; la otra celda no realiza ninguna acción.


In [ ]:
import os
import time
import json
import csv

CORRER_MICRO_REGISTRO = False

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

if MOTOR_REGISTRO == "valis":
    from valis import registration

    archivos_ok = all(
        os.path.exists(os.path.join(CARPETA_RECORTES, f"muestra_{i}.tif")) for i in range(1, 3)
    )
    if not archivos_ok:
        raise RuntimeError("Faltan muestra_1.tif / muestra_2.tif. Corré primero el Paso 1.")

    kwargs_valis = dict(
        src_dir=CARPETA_RECORTES,
        dst_dir=CARPETA_RESULTADOS,
        align_to_reference=True,
        reference_img_f="muestra_1.tif",
        crop="reference",
        max_processed_image_dim_px=VALIS_MAX_PROCESSED_IMG_DIM_PX,
        max_non_rigid_registration_dim_px=VALIS_MAX_NON_RIGID_DIM_PX,
        check_for_reflections=VALIS_CHECK_FOR_REFLECTIONS,
    )

    if VALIS_USAR_FEATURES_DEEP:
        try:
            from valis import feature_detectors, feature_matcher
            _fd = feature_detectors.DiskFD(rgb=True, num_features=VALIS_N_FEATURES)
            kwargs_valis["matcher"] = feature_matcher.LightGlueMatcher(_fd)
        except (ImportError, AttributeError):
            pass

    if VALIS_USAR_MICRO_RIGID_REGISTRAR:
        try:
            from valis.micro_rigid_registrar import MicroRigidRegistrar
            kwargs_valis["micro_rigid_registrar_cls"] = MicroRigidRegistrar
        except ImportError:
            pass

    if VALIS_NON_RIGID_REGISTRAR == "raft":
        try:
            from valis import non_rigid_registrars
            kwargs_valis["non_rigid_registrar_cls"] = non_rigid_registrars.RAFTWarper
        except (ImportError, AttributeError):
            pass

    _t0_registro = time.time()
    registrador = registration.Valis(**kwargs_valis)
    rigid_registrador, non_rigid_registrador, error_df = registrador.register()
    _seg_registro_base = time.time() - _t0_registro

    print(f"Referencia: {registrador.reference_img_f}")
    print(f"Tiempo de registro (rígido + no-rígido base): {_seg_registro_base:.0f} s")
    if error_df is not None:
        print(error_df.to_string())
    else:
        for nombre, slide in registrador.slide_dict.items():
            if hasattr(slide, "error_df") and slide.error_df is not None:
                fila = slide.error_df.iloc[0]
                print(f"{nombre}: original={fila.get('original_D', float('nan')):.1f} µm  "
                      f"rigid={fila.get('rigid_D', float('nan')):.1f} µm  "
                      f"non_rigid={fila.get('non_rigid_D', float('nan')):.1f} µm")

    _seg_micro = None
    micro_error_df = None
    if CORRER_MICRO_REGISTRO:
        _t0_micro = time.time()
        micro_reg, micro_error_df = registrador.register_micro(
            max_non_rigid_registration_dim_px=VALIS_MICRO_MAX_NON_RIGID_DIM_PX,
            align_to_reference=True
        )
        _seg_micro = time.time() - _t0_micro
        print(f"Tiempo de micro-registro: {_seg_micro:.0f} s")
        if micro_error_df is not None:
            print(micro_error_df.to_string())

        _fila_micro = {"timestamp": time.time()}
        if (error_df is not None and micro_error_df is not None
                and "mean_non_rigid_D" in error_df.columns
                and "mean_non_rigid_D" in micro_error_df.columns):
            antes = float(error_df["mean_non_rigid_D"].iloc[0])
            despues = float(micro_error_df["mean_non_rigid_D"].iloc[0])
            _fila_micro["mean_non_rigid_D_antes"] = antes
            _fila_micro["mean_non_rigid_D_despues"] = despues
            _fila_micro["empeoro"] = despues > antes
            print(f"mean_non_rigid_D: {antes:.1f} µm -> {despues:.1f} µm")

        _ruta_comparacion_micro = os.path.join(CARPETA_RESULTADOS, "comparacion_micro_registro.csv")
        _existe_csv_micro = os.path.exists(_ruta_comparacion_micro)
        with open(_ruta_comparacion_micro, "a", newline="") as _f:
            _writer_micro = csv.DictWriter(_f, fieldnames=list(_fila_micro.keys()))
            if not _existe_csv_micro:
                _writer_micro.writeheader()
            _writer_micro.writerow(_fila_micro)

    _ruta_tiempo = os.path.join(CARPETA_RESULTADOS, f"tiempo_registro_{MOTOR_REGISTRO}.json")
    with open(_ruta_tiempo, "w") as _f:
        json.dump({
            "motor": MOTOR_REGISTRO,
            "segundos_registro_base": _seg_registro_base,
            "segundos_micro_registro": _seg_micro,
            "segundos_total": _seg_registro_base + (_seg_micro or 0.0),
            "timestamp": time.time(),
        }, _f, indent=2)


In [ ]:
import os
import time
import json
import numpy as np

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

if MOTOR_REGISTRO == "tiatoolbox":
    from tiatoolbox.wsicore.wsireader import WSIReader
    from tiatoolbox.tools.registration.wsi_registration import (
        DFBRegister, AffineWSITransformer, match_histograms,
    )
    from skimage import color, exposure
    from skimage.filters import threshold_otsu
    from scipy import ndimage as _ndi
    from skimage import measure as _measure, morphology as _morph

    ruta_m1 = os.path.join(CARPETA_RECORTES, "muestra_1.tif")
    ruta_m2 = os.path.join(CARPETA_RECORTES, "muestra_2.tif")
    if not (os.path.exists(ruta_m1) and os.path.exists(ruta_m2)):
        raise RuntimeError(f"Faltan {ruta_m1} y/o {ruta_m2}. Corré primero el Paso 1.")

    if "mpp_base" not in globals() or mpp_base is None:
        with open(os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")) as _f:
            mpp_base = json.load(_f)["mpp_base"]

    lector1 = WSIReader.open(ruta_m1)
    lector2 = WSIReader.open(ruta_m2)
    ancho1, alto1 = lector1.info.slide_dimensions
    ancho2, alto2 = lector2.info.slide_dimensions

    mayor_dim_px = max(ancho1, alto1, ancho2, alto2)
    mpp_dfbr = mpp_base * (mayor_dim_px / TIATOOLBOX_DFBR_MAX_DIM_PX)

    fixed_rgb = lector1.slide_thumbnail(resolution=mpp_dfbr, units="mpp")
    moving_rgb = lector2.slide_thumbnail(resolution=mpp_dfbr, units="mpp")
    alto_fijo_thumb, ancho_fijo_thumb = fixed_rgb.shape[:2]
    alto_mov_thumb,  ancho_mov_thumb  = moving_rgb.shape[:2]

    def _preprocesar(img_rgb):
        g = color.rgb2gray(img_rgb)
        g = exposure.rescale_intensity(g, in_range=tuple(np.percentile(g, (0.5, 99.5))))
        return (g * 255).astype(np.uint8)

    fixed_gray = _preprocesar(fixed_rgb)
    moving_gray = _preprocesar(moving_rgb)
    fixed_gray, moving_gray = match_histograms(fixed_gray, moving_gray)

    def _mascara_tejido(gray):
        umbral = threshold_otsu(gray)
        m = (gray < umbral).astype(np.uint8)
        m = _ndi.binary_fill_holes(m, structure=np.ones((3, 3))).astype(np.uint8)
        etiquetas = _measure.label(m)
        if etiquetas.max() > 1:
            regiones = _measure.regionprops(etiquetas)
            mayor = max(r.area for r in regiones)
            m = _morph.remove_small_objects(m.astype(bool), min_size=int(mayor * 0.1)).astype(np.uint8)
        return m

    fixed_mask = _mascara_tejido(fixed_gray)
    moving_mask = _mascara_tejido(moving_gray)

    dfbr_fixed = np.repeat(np.expand_dims(fixed_gray, axis=2), 3, axis=2)
    dfbr_moving = np.repeat(np.expand_dims(moving_gray, axis=2), 3, axis=2)

    _t0_dfbr = time.time()
    dfbr = DFBRegister()
    dfbr_transform = dfbr.register(dfbr_fixed, dfbr_moving, fixed_mask, moving_mask)
    _seg_dfbr = time.time() - _t0_dfbr

    def _matriz_escala(sx, sy):
        return np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]], dtype=np.float64)

    escala_fija = _matriz_escala(ancho1 / ancho_fijo_thumb, alto1 / alto_fijo_thumb)
    escala_movil = _matriz_escala(ancho2 / ancho_mov_thumb, alto2 / alto_mov_thumb)
    transform_level0 = escala_fija @ dfbr_transform @ np.linalg.inv(escala_movil)

    transformador_afin = AffineWSITransformer(lector2, transform_level0)

    np.save(os.path.join(CARPETA_RESULTADOS, "transform_level0_tiatoolbox.npy"), transform_level0)
    with open(os.path.join(CARPETA_RESULTADOS, "tiatoolbox_registro_metadata.json"), "w") as _f:
        json.dump({
            "mpp_dfbr": float(mpp_dfbr),
            "ancho1": int(ancho1), "alto1": int(alto1),
            "ancho2": int(ancho2), "alto2": int(alto2),
            "timestamp": time.time(),
        }, _f, indent=2)

    _ruta_tiempo = os.path.join(CARPETA_RESULTADOS, f"tiempo_registro_{MOTOR_REGISTRO}.json")
    with open(_ruta_tiempo, "w") as _f:
        json.dump({
            "motor": MOTOR_REGISTRO,
            "segundos_registro_base": _seg_dfbr,
            "segundos_micro_registro": None,
            "segundos_total": _seg_dfbr,
            "timestamp": time.time(),
        }, _f, indent=2)

    print(f"muestra_1: {ancho1}x{alto1} px | muestra_2: {ancho2}x{alto2} px | mpp_base={mpp_base:.4f}")
    print(f"Registro DFBR completado en {_seg_dfbr:.0f} s")
    print(f"Transform (nivel 0):\n{transform_level0}")


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

TAMANIO_VIS_MAX_PX = 2000

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

if MOTOR_REGISTRO == "valis":
    slide1 = registrador.get_slide("muestra_1")
    slide2 = registrador.get_slide("muestra_2")

    def _vips_a_miniatura_numpy(img_vips, tamanio_max=TAMANIO_VIS_MAX_PX):
        lado_mayor = max(img_vips.width, img_vips.height)
        escala = min(1.0, tamanio_max / lado_mayor)
        if escala < 1.0:
            img_vips = img_vips.resize(escala)
        return img_vips.numpy()

    img_ref = _vips_a_miniatura_numpy(slide1.slide2vips(level=0))
    axes[0].imshow(img_ref)
    axes[0].set_title(f"Referencia: {slide1.name}")
    axes[0].axis("off")

    img_alineada = _vips_a_miniatura_numpy(slide2.warp_slide(level=0))
    axes[1].imshow(img_alineada)
    axes[1].set_title(f"Alineada: {slide2.name}")
    axes[1].axis("off")

    ruta_overlap = os.path.join(CARPETA_RESULTADOS, "overlap", f"{registrador.name}_overlap.png")
    if os.path.exists(ruta_overlap):
        overlap = plt.imread(ruta_overlap)
        axes[2].imshow(overlap)
        axes[2].set_title("Superposición")
    axes[2].axis("off")

elif MOTOR_REGISTRO == "tiatoolbox":
    import cv2
    axes[0].imshow(fixed_rgb)
    axes[0].set_title("Referencia: muestra_1 (miniatura DFBR)")
    axes[0].axis("off")

    moving_warp = cv2.warpAffine(
        moving_gray, dfbr_transform[0:-1], fixed_gray.shape[:2][::-1]
    )
    axes[1].imshow(moving_warp, cmap="gray")
    axes[1].set_title("Alineada: muestra_2 (afín DFBR, miniatura)")
    axes[1].axis("off")

    # Vista previa del registro afín a resolución de miniatura; el
    # refinamiento B-spline (Paso 3) se aplica por tile y no se refleja acá.
    overlay = np.dstack((moving_warp, fixed_gray, moving_warp))
    axes[2].imshow(overlay)
    axes[2].set_title("Superposición")
    axes[2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, f"visualizacion_alineacion_{MOTOR_REGISTRO}.png"),
            dpi=150, bbox_inches="tight")
plt.show()


---
## Paso 2.5 — Campo de deformación de VALIS (solo `valis`)


In [ ]:
import os
import json
import numpy as np
import pandas as pd

df_dxdy_grid = None

if MOTOR_REGISTRO == "valis":
    if "mpp_base" not in globals():
        with open(os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")) as _f:
            mpp_base = json.load(_f)["mpp_base"]

    def _campo_desplazamiento_no_rigido(slide):
        # Distintas versiones de VALIS exponen el campo no-rígido bajo
        # nombres distintos; se prueban los dos conocidos.
        for nombre_attr in ("bk_dxdy", "fwd_dxdy"):
            campo = getattr(slide, nombre_attr, None)
            if campo is None:
                continue
            campo = np.asarray(campo)
            if campo.ndim == 3 and campo.shape[0] == 2:
                return campo[0], campo[1], nombre_attr
            if campo.ndim == 3 and campo.shape[-1] == 2:
                return campo[..., 0], campo[..., 1], nombre_attr
        return None, None, None

    dx_proc, dy_proc, atributo_dxdy_usado = _campo_desplazamiento_no_rigido(slide2)

    if dx_proc is not None:
        alto_proc, ancho_proc = dx_proc.shape
        ancho_full, alto_full = slide2.slide_dimensions_wh[NIVEL_TILES]

        factor_x = ancho_full / ancho_proc
        factor_y = alto_full / alto_proc

        dx_full = dx_proc * factor_x
        dy_full = dy_proc * factor_y
        magnitud_proc = np.hypot(dx_full, dy_full)

        posiciones_x_dxdy = list(range(0, ancho_full - TILE_SIZE + 1, STRIDE))
        posiciones_y_dxdy = list(range(0, alto_full  - TILE_SIZE + 1, STRIDE))

        registros_dxdy = []
        for fila, y0_dxdy in enumerate(posiciones_y_dxdy):
            for col, x0_dxdy in enumerate(posiciones_x_dxdy):
                cx_full = x0_dxdy + TILE_SIZE / 2
                cy_full = y0_dxdy + TILE_SIZE / 2
                cx_proc = min(int(cx_full / factor_x), ancho_proc - 1)
                cy_proc = min(int(cy_full / factor_y), alto_proc - 1)
                registros_dxdy.append({
                    "fila":        fila,
                    "col":         col,
                    "x1":          x0_dxdy,
                    "y1":          y0_dxdy,
                    "dxdy_mag_px": float(magnitud_proc[cy_proc, cx_proc]),
                })

        df_dxdy_grid = pd.DataFrame(registros_dxdy)
        if mpp_base is not None:
            df_dxdy_grid["dxdy_mag_um"] = df_dxdy_grid["dxdy_mag_px"] * mpp_base

        os.makedirs(CARPETA_TILES, exist_ok=True)
        ruta_dxdy_csv = os.path.join(CARPETA_TILES, "metrica_dxdy_valis.csv")
        df_dxdy_grid.to_csv(ruta_dxdy_csv, index=False)

        print(f"Campo no-rígido: slide2.{atributo_dxdy_usado}, shape={dx_proc.shape}")
        print(f"Resolución completa (nivel {NIVEL_TILES}): {ancho_full}x{alto_full} px")
        if "dxdy_mag_um" in df_dxdy_grid.columns:
            print(df_dxdy_grid["dxdy_mag_um"].describe().round(1))


---
## Paso 3 — Extracción de tiles pareados


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import cv2

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

carpeta_tiles_1 = os.path.join(CARPETA_TILES, "muestra_1")
carpeta_tiles_2 = os.path.join(CARPETA_TILES, "muestra_2")
os.makedirs(carpeta_tiles_1, exist_ok=True)
os.makedirs(carpeta_tiles_2, exist_ok=True)

ruta_metadata_pipeline = os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")

if MOTOR_REGISTRO == "valis":
    slide_ref    = registrador.get_slide("muestra_1")
    slide_moving = registrador.get_slide("muestra_2")

    img_ref_completa      = slide_ref.warp_slide(level=NIVEL_TILES, crop="reference").numpy()
    img_alineada_completa = slide_moving.warp_slide(level=NIVEL_TILES, crop="reference").numpy()

    alto_ref,  ancho_ref  = img_ref_completa.shape[:2]
    alto_mov,  ancho_mov  = img_alineada_completa.shape[:2]
    alto_ref  = min(alto_ref,  alto_mov)
    ancho_ref = min(ancho_ref, ancho_mov)
    img_ref_completa      = img_ref_completa[:alto_ref, :ancho_ref]
    img_alineada_completa = img_alineada_completa[:alto_ref, :ancho_ref]

    def _leer_tile_par(x1, y1):
        y2, x2 = y1 + TILE_SIZE, x1 + TILE_SIZE
        tile_ref = img_ref_completa[y1:y2, x1:x2]
        h_moving, w_moving = img_alineada_completa.shape[:2]
        if y2 > h_moving or x2 > w_moving:
            tile_moving = np.full((TILE_SIZE, TILE_SIZE, 3), 255, dtype=img_alineada_completa.dtype)
        else:
            tile_moving = img_alineada_completa[y1:y2, x1:x2]
        return tile_ref, tile_moving

    _tiempo_bspline_total = None

elif MOTOR_REGISTRO == "tiatoolbox":
    import time
    from tiatoolbox.wsicore.wsireader import WSIReader
    from tiatoolbox.tools.registration.wsi_registration import (
        AffineWSITransformer, estimate_bspline_transform, apply_bspline_transform,
    )

    if "lector1" not in globals() or "transformador_afin" not in globals():
        ruta_m1 = os.path.join(CARPETA_RECORTES, "muestra_1.tif")
        ruta_m2 = os.path.join(CARPETA_RECORTES, "muestra_2.tif")
        lector1 = WSIReader.open(ruta_m1)
        lector2 = WSIReader.open(ruta_m2)
        transform_level0 = np.load(os.path.join(CARPETA_RESULTADOS, "transform_level0_tiatoolbox.npy"))
        transformador_afin = AffineWSITransformer(lector2, transform_level0)

    ancho_ref, alto_ref = lector1.info.slide_dimensions

    _grid_space_px = TIATOOLBOX_BSPLINE_GRID_SPACE_UM / mpp_base if mpp_base else 200.0
    _tiempo_bspline_total = 0.0
    _n_bspline_ok, _n_bspline_fallo = 0, 0

    def _refinar_bspline(tile_ref, tile_moving):
        global _tiempo_bspline_total, _n_bspline_ok, _n_bspline_fallo
        _t0 = time.time()
        try:
            mask_ref = np.ones(tile_ref.shape[:2], dtype=int)
            mask_moving = np.ones(tile_moving.shape[:2], dtype=int)
            bspline_tf = estimate_bspline_transform(
                tile_ref, tile_moving, mask_ref, mask_moving,
                grid_space=_grid_space_px, sampling_percent=TIATOOLBOX_BSPLINE_SAMPLING_PERCENT,
            )
            tile_refinado = apply_bspline_transform(tile_ref, tile_moving, bspline_tf)
            _n_bspline_ok += 1
            _tiempo_bspline_total += time.time() - _t0
            return tile_refinado
        except Exception:
            _n_bspline_fallo += 1
            _tiempo_bspline_total += time.time() - _t0
            return tile_moving

    def _leer_tile_par(x1, y1):
        tile_ref = lector1.read_rect((x1, y1), (TILE_SIZE, TILE_SIZE), resolution=NIVEL_TILES, units="level")
        try:
            tile_moving = transformador_afin.read_rect(
                (x1, y1), (TILE_SIZE, TILE_SIZE), resolution=NIVEL_TILES, units="level"
            )
        except Exception:
            tile_moving = np.full((TILE_SIZE, TILE_SIZE, 3), 255, dtype=np.uint8)

        if tile_ref.shape[:2] != (TILE_SIZE, TILE_SIZE):
            tile_ref = cv2.resize(tile_ref, (TILE_SIZE, TILE_SIZE))
        if tile_moving.shape[:2] != (TILE_SIZE, TILE_SIZE):
            tile_moving = cv2.resize(tile_moving, (TILE_SIZE, TILE_SIZE))

        if TIATOOLBOX_REFINAR_BSPLINE_POR_TILE:
            tile_moving = _refinar_bspline(tile_ref, tile_moving)
        return tile_ref, tile_moving

else:
    raise RuntimeError(f"MOTOR_REGISTRO desconocido: {MOTOR_REGISTRO!r}")

posiciones_x = range(0, ancho_ref - TILE_SIZE + 1, STRIDE)
posiciones_y = range(0, alto_ref  - TILE_SIZE + 1, STRIDE)

registros = []
tiles_guardados = 0
tiles_descartados = 0


def medir_informatividad(img_rgb, mask=None):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    lap  = cv2.Laplacian(gray, cv2.CV_64F)
    if mask is not None and np.sum(mask > 0) > 100:
        return float(lap[mask > 0].var())
    return float(lap.var())


_lista_x = list(posiciones_x)
_lista_y = list(posiciones_y)

for fila, y1 in enumerate(tqdm(_lista_y, desc="Extrayendo tiles")):
    for col, x1 in enumerate(_lista_x):
        y2 = y1 + TILE_SIZE
        x2 = x1 + TILE_SIZE

        tile_ref, tile_moving = _leer_tile_par(x1, y1)

        gris = np.mean(tile_ref, axis=2) if tile_ref.ndim == 3 else tile_ref
        fraccion_tejido = np.mean(gris < 240)

        if fraccion_tejido < UMBRAL_TEJIDO:
            tiles_descartados += 1
            continue

        if medir_informatividad(tile_ref) < MIN_INFORMATIVIDAD:
            tiles_descartados += 1
            continue

        nombre_tile = f"tile_r{y1:06d}_c{x1:06d}.png"
        ruta_tile_1 = os.path.join(carpeta_tiles_1, nombre_tile)
        ruta_tile_2 = os.path.join(carpeta_tiles_2, nombre_tile)

        cv2.imwrite(ruta_tile_1, cv2.cvtColor(tile_ref,    cv2.COLOR_RGB2BGR))
        cv2.imwrite(ruta_tile_2, cv2.cvtColor(tile_moving, cv2.COLOR_RGB2BGR))

        registros.append({
            "nombre": nombre_tile,
            "fila": fila,
            "col": col,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "fraccion_tejido": round(fraccion_tejido, 3)
        })
        tiles_guardados += 1

df_tiles = pd.DataFrame(registros)

try:
    with open(ruta_metadata_pipeline, "r") as _f:
        _metadata_offset = json.load(_f)
    _off = _metadata_offset["offsets_recorte"]["muestra_1"]
    df_tiles["x1_orig"] = df_tiles["x1"] + _off["x"]
    df_tiles["y1_orig"] = df_tiles["y1"] + _off["y"]
except (FileNotFoundError, KeyError):
    pass

ruta_csv = os.path.join(CARPETA_TILES, "indice_tiles.csv")
df_tiles.to_csv(ruta_csv, index=False)

if MOTOR_REGISTRO == "tiatoolbox" and TIATOOLBOX_REFINAR_BSPLINE_POR_TILE:
    _ruta_tiempo = os.path.join(CARPETA_RESULTADOS, f"tiempo_registro_{MOTOR_REGISTRO}.json")
    if os.path.exists(_ruta_tiempo):
        with open(_ruta_tiempo) as _f:
            _tiempos = json.load(_f)
        _tiempos["segundos_micro_registro"] = _tiempo_bspline_total
        _tiempos["segundos_total"] = _tiempos.get("segundos_registro_base", 0.0) + _tiempo_bspline_total
        with open(_ruta_tiempo, "w") as _f:
            json.dump(_tiempos, _f, indent=2)

print(f"Dimensiones de referencia: {ancho_ref} x {alto_ref} px")
print(f"Tiles guardados: {tiles_guardados} | descartados: {tiles_descartados}")
print(f"Índice: {ruta_csv}")
if MOTOR_REGISTRO == "tiatoolbox" and TIATOOLBOX_REFINAR_BSPLINE_POR_TILE:
    print(f"Refinamiento B-spline por tile: {_n_bspline_ok} ok, {_n_bspline_fallo} fallidos, "
          f"{_tiempo_bspline_total:.0f} s")


In [ ]:
import os
import random
import cv2
import pandas as pd
import matplotlib.pyplot as plt

random.seed(SEMILLA)
n_muestra = min(4, tiles_guardados)
muestra_tiles = random.sample(registros, n_muestra)

fig, axes = plt.subplots(n_muestra, 2, figsize=(8, n_muestra * 4))
if n_muestra == 1:
    axes = [axes]

for i, info in enumerate(muestra_tiles):
    nombre = info["nombre"]

    tile1 = cv2.cvtColor(cv2.imread(os.path.join(carpeta_tiles_1, nombre)), cv2.COLOR_BGR2RGB)
    tile2 = cv2.cvtColor(cv2.imread(os.path.join(carpeta_tiles_2, nombre)), cv2.COLOR_BGR2RGB)

    axes[i][0].imshow(tile1)
    axes[i][0].set_title(f"Muestra 1\n{nombre}")
    axes[i][0].axis("off")

    axes[i][1].imshow(tile2)
    axes[i][1].set_title(f"Muestra 2 (alineada)\n{nombre}")
    axes[i][1].axis("off")

plt.suptitle("Pares de tiles extraídos (misma región en ambas muestras)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_TILES, "muestra_tiles.png"), dpi=150, bbox_inches="tight")
plt.show()

df = pd.read_csv(os.path.join(CARPETA_TILES, "indice_tiles.csv"))
df[["x1", "y1", "x1_orig", "y1_orig"]].head()


---
## Paso 4 — Métricas de calidad por tile


In [ ]:
import os
import json
import warnings
import pyvips
import pandas as pd

carpeta_tiles_1 = os.path.join(CARPETA_TILES, "muestra_1")
carpeta_tiles_2 = os.path.join(CARPETA_TILES, "muestra_2")

if "registros" not in globals() or "df_tiles" not in globals():
    ruta_csv_tiles = os.path.join(CARPETA_TILES, "indice_tiles.csv")
    df_tiles = pd.read_csv(ruta_csv_tiles)
    registros = df_tiles.to_dict("records")

if "mpp_base" not in globals() or mpp_base is None:
    ruta_meta = os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")
    with open(ruta_meta) as f:
        mpp_base = json.load(f)["mpp_base"]

_ruta_meta_pipeline = os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")
_ruta_csv = os.path.join(CARPETA_TILES, "indice_tiles.csv")
if os.path.exists(_ruta_meta_pipeline) and os.path.exists(_ruta_csv):
    if os.path.getmtime(_ruta_meta_pipeline) > os.path.getmtime(_ruta_csv):
        warnings.warn("El Paso 1 se corrió después del Paso 3; se recomienda repetir el Paso 3.")

print(f"Tiles cargados: {len(registros)}  |  mpp_base: {mpp_base:.4f} µm/px")


In [ ]:
import cv2
import numpy as np
import random

_clahe_op = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def metrica_similitud_tiles(tile1, tile2, mpp=None, ventana=True,
                             modo="gradiente", usar_clahe=True):
    g1 = cv2.cvtColor(tile1, cv2.COLOR_RGB2GRAY) if tile1.ndim == 3 else tile1.copy()
    g2 = cv2.cvtColor(tile2, cv2.COLOR_RGB2GRAY) if tile2.ndim == 3 else tile2.copy()

    if modo == "gradiente":
        if usar_clahe:
            g1 = _clahe_op.apply(g1)
            g2 = _clahe_op.apply(g2)
        gx1 = cv2.Sobel(g1, cv2.CV_32F, 1, 0, ksize=3)
        gy1 = cv2.Sobel(g1, cv2.CV_32F, 0, 1, ksize=3)
        f1 = cv2.magnitude(gx1, gy1)

        gx2 = cv2.Sobel(g2, cv2.CV_32F, 1, 0, ksize=3)
        gy2 = cv2.Sobel(g2, cv2.CV_32F, 0, 1, ksize=3)
        f2 = cv2.magnitude(gx2, gy2)
    elif modo == "crudo":
        f1, f2 = g1.astype(np.float32), g2.astype(np.float32)
    else:
        raise ValueError("modo debe ser 'gradiente' o 'crudo'")

    win = cv2.createHanningWindow((f1.shape[1], f1.shape[0]), cv2.CV_32F) if ventana else None
    (dx, dy), response = cv2.phaseCorrelate(f1, f2, win)
    shift_mag_px = float(np.hypot(dx, dy))

    resultado = {
        "shift_x_px": dx,
        "shift_y_px": dy,
        "shift_mag_px": shift_mag_px,
        "phase_corr_response": float(response),
    }
    if mpp is not None:
        resultado["shift_mag_um"] = shift_mag_px * mpp
    return resultado

def metrica_ecc_tiles(tile1, tile2, mpp=None, max_iter=200, eps=1e-6, shift_inicial=None):
    g1 = cv2.cvtColor(tile1, cv2.COLOR_RGB2GRAY) if tile1.ndim == 3 else tile1
    g2 = cv2.cvtColor(tile2, cv2.COLOR_RGB2GRAY) if tile2.ndim == 3 else tile2
    g1, g2 = g1.astype(np.float32), g2.astype(np.float32)

    warp = np.eye(2, 3, dtype=np.float32)
    if shift_inicial is not None:
        warp[0, 2] = float(shift_inicial[0])
        warp[1, 2] = float(shift_inicial[1])
    criterio = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, max_iter, eps)
    try:
        ecc_score, warp = cv2.findTransformECC(
            g1, g2, warp, cv2.MOTION_TRANSLATION, criterio
        )
        dx, dy = warp[0, 2], warp[1, 2]
        shift_mag_px = float(np.hypot(dx, dy))
        resultado = {
            "shift_x_px_ecc": dx, "shift_y_px_ecc": dy,
            "shift_mag_px_ecc": shift_mag_px, "ecc_score": float(ecc_score),
        }
        if mpp is not None:
            resultado["shift_mag_um_ecc"] = shift_mag_px * mpp
        return resultado
    except cv2.error:
        return {
            "shift_x_px_ecc": np.nan, "shift_y_px_ecc": np.nan,
            "shift_mag_px_ecc": np.nan, "ecc_score": None,
            **({"shift_mag_um_ecc": np.nan} if mpp is not None else {}),
        }

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

def leer_par_tiles(nombre, carpeta_1, carpeta_2):
    ruta_1 = os.path.join(carpeta_1, nombre)
    ruta_2 = os.path.join(carpeta_2, nombre)
    img1 = cv2.imread(ruta_1)
    img2 = cv2.imread(ruta_2)
    if img1 is None or img2 is None:
        return None, None
    return cv2.cvtColor(img1, cv2.COLOR_BGR2RGB), cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
n_grilla = 6
UMBRAL_STD = 5.0

alto, ancho = img_ref_completa.shape[:2]
paso_y, paso_x = alto // n_grilla, ancho // n_grilla
resultados_grilla = []
saltados_por_std = 0

for i in range(1, n_grilla):
    for j in range(1, n_grilla):
        y, x = i * paso_y, j * paso_x
        parche_ref = img_ref_completa[y:y + TILE_SIZE, x:x + TILE_SIZE]
        parche_mov = img_alineada_completa[y:y + TILE_SIZE, x:x + TILE_SIZE]
        if parche_ref.shape[:2] != (TILE_SIZE, TILE_SIZE) or parche_mov.shape[:2] != (TILE_SIZE, TILE_SIZE):
            continue
        if np.mean(parche_ref) > 245:
            continue
        if np.std(cv2.cvtColor(parche_ref, cv2.COLOR_RGB2GRAY)) < UMBRAL_STD:
            saltados_por_std += 1
            continue
        m = metrica_similitud_tiles(parche_ref, parche_mov, mpp=mpp_base)
        m["x"], m["y"] = x, y
        resultados_grilla.append(m)

df_grilla = pd.DataFrame(resultados_grilla)

# Un shift promedio grande y consistente en un solo signo indica offset
# global entre las dos muestras, no error local de registro.
print(f"Parches evaluados: {len(df_grilla)} ({saltados_por_std} descartados por baja textura)")
print(f"Shift promedio: x={df_grilla['shift_x_px'].mean():.2f} px, "
      f"y={df_grilla['shift_y_px'].mean():.2f} px")
print(f"phase_corr_response promedio: {df_grilla['phase_corr_response'].mean():.3f}")

df_grilla[["x", "y", "shift_x_px", "shift_y_px", "shift_mag_um", "phase_corr_response"]].round(2)


In [ ]:
resultados_metrica = []
tiles_no_leidos = []
for info in tqdm(registros, desc="Calculando métricas"):
    tile1, tile2 = leer_par_tiles(info["nombre"], carpeta_tiles_1, carpeta_tiles_2)
    if tile1 is None:
        tiles_no_leidos.append(info["nombre"])
        continue
    m = metrica_similitud_tiles(tile1, tile2, mpp=mpp_base)
    m["nombre"] = info["nombre"]
    resultados_metrica.append(m)

df_metricas = pd.DataFrame(resultados_metrica)
df_metricas["confianza_pct"] = df_metricas["phase_corr_response"].rank(pct=True) * 100

df_tiles_completo = df_tiles.merge(df_metricas, on="nombre")

ruta_csv_metricas = os.path.join(CARPETA_TILES, "indice_tiles_con_metricas.csv")
df_tiles_completo.to_csv(ruta_csv_metricas, index=False)

print(f"Métricas calculadas para {len(df_tiles_completo)} pares de tiles "
      f"({len(tiles_no_leidos)} no se pudieron leer)")
df_tiles_completo[["shift_mag_px", "shift_mag_um", "phase_corr_response"]].describe().round(3)


---
## Paso 4.5 — Ajuste ECC post-hoc


In [ ]:
import random as _random

MOTOR_LABEL = "VALIS" if MOTOR_REGISTRO == "valis" else "tiatoolbox"


def ecc_post_hoc(tile1_rgb, tile2_rgb, motion=cv2.MOTION_EUCLIDEAN, max_iter=200, eps=1e-6, shift_inicial=None):
    g1 = cv2.cvtColor(tile1_rgb, cv2.COLOR_RGB2GRAY) if tile1_rgb.ndim == 3 else tile1_rgb
    g2 = cv2.cvtColor(tile2_rgb, cv2.COLOR_RGB2GRAY) if tile2_rgb.ndim == 3 else tile2_rgb
    warp_matrix = np.eye(2, 3, dtype=np.float32)
    if shift_inicial is not None:
        warp_matrix[0, 2] = float(shift_inicial[0])
        warp_matrix[1, 2] = float(shift_inicial[1])
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, max_iter, eps)
    try:
        _, warp_matrix = cv2.findTransformECC(
            g1.astype(np.float32), g2.astype(np.float32),
            warp_matrix, motion, criteria, None, 5
        )
    except cv2.error:
        return tile2_rgb, False

    alto, ancho = tile1_rgb.shape[:2]
    tile2_ajustado = cv2.warpAffine(
        tile2_rgb, warp_matrix, (ancho, alto),
        flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
        borderMode=cv2.BORDER_REPLICATE
    )
    return tile2_ajustado, True


N_MUESTRA_ECC_POSTHOC = None
_semilla_ecc = SEMILLA if "SEMILLA" in globals() else 42

nombres_ecc = df_tiles_completo["nombre"].tolist()
if N_MUESTRA_ECC_POSTHOC is not None and len(nombres_ecc) > N_MUESTRA_ECC_POSTHOC:
    _rng = _random.Random(_semilla_ecc)
    nombres_ecc = _rng.sample(nombres_ecc, N_MUESTRA_ECC_POSTHOC)

_shift_por_nombre = df_tiles_completo.set_index("nombre")[["shift_x_px", "shift_y_px"]].to_dict("index")

resultados_posthoc = []
n_convergio_posthoc = 0
for nombre in tqdm(nombres_ecc, desc=f"ECC post-hoc ({MOTOR_LABEL})"):
    tile1, tile2 = leer_par_tiles(nombre, carpeta_tiles_1, carpeta_tiles_2)
    if tile1 is None:
        continue
    _shift_pc = _shift_por_nombre.get(nombre)
    _shift_inicial = (_shift_pc["shift_x_px"], _shift_pc["shift_y_px"]) if _shift_pc else None
    tile2_ajustado, convergio = ecc_post_hoc(tile1, tile2, shift_inicial=_shift_inicial)
    n_convergio_posthoc += int(convergio)
    m = metrica_similitud_tiles(tile1, tile2_ajustado, mpp=mpp_base)
    m["nombre"] = nombre
    # 'ecc': ECC convergió en el tile; 'geom': se usó el shift geométrico
    # inicial como fallback (requerido para comparacionPipelines_paper.ipynb).
    m["categoria_ecc"] = "ecc" if convergio else "geom"
    resultados_posthoc.append(m)

df_posthoc_motor_actual = pd.DataFrame(resultados_posthoc)

_comparacion = pd.concat([
    df_tiles_completo.loc[df_tiles_completo["nombre"].isin(nombres_ecc), ["nombre", "shift_mag_um"]]
        .assign(metodo=f"{MOTOR_LABEL} (bruto)"),
    df_posthoc_motor_actual[["nombre", "shift_mag_um"]].assign(
        metodo=f"{MOTOR_LABEL} + ECC (" + df_posthoc_motor_actual["categoria_ecc"] + ")"
    ),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 5))
_metodos = _comparacion["metodo"].unique().tolist()
_datos = [_comparacion.loc[_comparacion["metodo"] == m, "shift_mag_um"].dropna() for m in _metodos]
ax.boxplot(_datos, tick_labels=_metodos, showfliers=False)
ax.set_ylabel("Desplazamiento residual (µm)")
ax.set_title(f"{MOTOR_LABEL}: bruto vs. con ajuste ECC por tile")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_TILES, "boxplot_ecc_posthoc.png"), dpi=150, bbox_inches="tight")
plt.show()

ruta_csv_posthoc = os.path.join(CARPETA_TILES, "ecc_posthoc.csv")
df_posthoc_motor_actual.to_csv(ruta_csv_posthoc, index=False)

_sufijo_muestra_ecc = "completo" if N_MUESTRA_ECC_POSTHOC is None else f"muestra{N_MUESTRA_ECC_POSTHOC}"
ruta_csv_posthoc_tagged = os.path.join(CARPETA_TILES, f"ecc_posthoc_{_sufijo_muestra_ecc}.csv")
df_posthoc_motor_actual.to_csv(ruta_csv_posthoc_tagged, index=False)

print(f"ECC convergió en {n_convergio_posthoc}/{len(df_posthoc_motor_actual)} tiles "
      f"({100 * n_convergio_posthoc / len(df_posthoc_motor_actual):.1f}%)")
_comparacion.groupby("metodo")["shift_mag_um"].median().round(1).sort_values()


---
## Paso 5 — Mapa espacial de zonas problemáticas


In [ ]:
import matplotlib.patches as patches
from PIL import Image

MINI = 20

n_filas_grid = df_tiles_completo["fila"].max() + 1
n_cols_grid  = df_tiles_completo["col"].max() + 1

mosaico = np.full((n_filas_grid * MINI, n_cols_grid * MINI, 3), 255, dtype=np.uint8)

for _, fila in df_tiles_completo.iterrows():
    ruta_tile = os.path.join(carpeta_tiles_1, fila["nombre"])
    if not os.path.exists(ruta_tile):
        continue
    tile_chico = np.array(Image.open(ruta_tile).convert("RGB").resize((MINI, MINI)))
    f, c = int(fila["fila"]), int(fila["col"])
    mosaico[f*MINI:(f+1)*MINI, c*MINI:(c+1)*MINI] = tile_chico

UMBRAL = df_tiles_completo["shift_mag_um"].quantile(0.95)

fallos = df_tiles_completo[
    (df_tiles_completo["phase_corr_response"] < UMBRAL_RESPONSE_FALLA) &
    (df_tiles_completo["shift_mag_um"] > df_tiles_completo["shift_mag_um"].median() * UMBRAL_SHIFT_FALLA_RELATIVO)
]
peores = df_tiles_completo[df_tiles_completo["shift_mag_um"] >= UMBRAL].drop(fallos.index, errors="ignore")

ANCHO_BANDA = 7
fila_min, fila_max = df_tiles_completo["fila"].min(), df_tiles_completo["fila"].max()
mejor_inicio, mejor_conteo = fila_min, -1
for inicio in range(fila_min, fila_max - ANCHO_BANDA + 2):
    conteo = ((peores["fila"] >= inicio) & (peores["fila"] < inicio + ANCHO_BANDA)).sum()
    if conteo > mejor_conteo:
        mejor_conteo, mejor_inicio = conteo, inicio

zona_filas = (mejor_inicio, mejor_inicio + ANCHO_BANDA - 1)

fig, ax = plt.subplots(figsize=(13, max(6, 13 * n_filas_grid / n_cols_grid)))
ax.imshow(mosaico)

rect = patches.Rectangle(
    (0, zona_filas[0] * MINI), n_cols_grid * MINI, ANCHO_BANDA * MINI,
    linewidth=2, edgecolor="red", facecolor="red", alpha=0.15
)
ax.add_patch(rect)
ax.text(5, zona_filas[0] * MINI - 5, "zona con más tiles mal alineados",
        color="red", fontsize=10, weight="bold", va="bottom")

for _, fila in peores.iterrows():
    cx = fila["col"] * MINI + MINI / 2
    cy = fila["fila"] * MINI + MINI / 2
    ax.add_patch(patches.Circle((cx, cy), MINI * 0.7, edgecolor="black", facecolor="none", linewidth=1.3))

for _, fila in fallos.iterrows():
    cx = fila["col"] * MINI + MINI / 2
    cy = fila["fila"] * MINI + MINI / 2
    ax.plot(cx, cy, marker="x", color="cyan", markersize=14, markeredgewidth=3)

ax.set_title("Mapa de calidad de alineación sobre el tejido real\n"
             "(círculos negros = 5% peor por desplazamiento residual   ·   X celeste = falla de medición)")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_TILES, "mapa_zonas_problematicas.png"), dpi=150, bbox_inches="tight")
plt.show()

N_ZOOM = 6
zona_tiles = (
    peores[(peores["fila"] >= zona_filas[0]) & (peores["fila"] <= zona_filas[1])]
    .sort_values("shift_mag_um", ascending=False)
    .head(N_ZOOM)
)

if len(zona_tiles) > 0:
    fig, axes = plt.subplots(len(zona_tiles), 2, figsize=(7, len(zona_tiles) * 3.3))
    if len(zona_tiles) == 1:
        axes = [axes]
    for i, (_, fila) in enumerate(zona_tiles.iterrows()):
        tile1 = cv2.cvtColor(cv2.imread(os.path.join(carpeta_tiles_1, fila["nombre"])), cv2.COLOR_BGR2RGB)
        tile2 = cv2.cvtColor(cv2.imread(os.path.join(carpeta_tiles_2, fila["nombre"])), cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(tile1); axes[i][0].axis("off")
        axes[i][0].set_title(f"Muestra 1\nfila {int(fila['fila'])}, col {int(fila['col'])}", fontsize=9)
        axes[i][1].imshow(tile2); axes[i][1].axis("off")
        axes[i][1].set_title(
            f"Muestra 2\nshift={fila['shift_mag_um']:.0f}µm   resp={fila['phase_corr_response']:.2f}",
            fontsize=9
        )

    plt.suptitle(f"Tiles reales dentro de la zona marcada (filas {zona_filas[0]}-{zona_filas[1]})", y=1.0)
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_TILES, "zoom_zona_problematica.png"), dpi=150, bbox_inches="tight")
    plt.show()

print(f"Umbral (percentil 95): {UMBRAL:.1f} µm | {len(peores)} tiles peores | "
      f"{len(fallos)} fallos de medición")
print(f"Banda con mayor concentración de tiles problemáticos: filas {zona_filas[0]}-{zona_filas[1]} "
      f"({mejor_conteo}/{len(peores)})")


In [ ]:
import os
import pandas as pd
import cv2
import numpy as np

carpeta_tiles_1 = os.path.join(CARPETA_TILES, "muestra_1")
carpeta_tiles_2 = os.path.join(CARPETA_TILES, "muestra_2")

ruta_csv_metricas = os.path.join(CARPETA_TILES, "indice_tiles_con_metricas.csv")
if "df_tiles_completo" not in globals():
    df_tiles_completo = pd.read_csv(ruta_csv_metricas)

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def fraccion_tejido_de(ruta_png):
    img = cv2.imread(ruta_png)
    if img is None:
        return np.nan
    gris = np.mean(img, axis=2)
    return float(np.mean(gris < 240))


fracciones_2 = []
for nombre in tqdm(df_tiles_completo["nombre"], desc="Fracción de tejido (muestra 2)"):
    ruta = os.path.join(carpeta_tiles_2, nombre)
    fracciones_2.append(fraccion_tejido_de(ruta))

df_tiles_completo["fraccion_tejido_m2"] = fracciones_2

UMBRAL_FALLA_MEDICION_UM = df_tiles_completo["shift_mag_um"].quantile(PERCENTIL_FALLA_MEDICION)
mediana_shift_global = df_tiles_completo["shift_mag_um"].median()

df_clean = df_tiles_completo[df_tiles_completo["shift_mag_um"] < UMBRAL_FALLA_MEDICION_UM].copy()
p95_shift = df_clean["shift_mag_um"].quantile(0.95)


def clasificar(fila):
    es_shift_extremo = fila["shift_mag_um"] >= UMBRAL_FALLA_MEDICION_UM
    es_response_nula_y_shift_absurdo = (
        fila["phase_corr_response"] < UMBRAL_RESPONSE_FALLA
        and fila["shift_mag_um"] > mediana_shift_global * UMBRAL_SHIFT_FALLA_RELATIVO
    )
    if es_shift_extremo or es_response_nula_y_shift_absurdo:
        return "falla_medicion"

    tiene_tejido_1 = fila["fraccion_tejido"] >= UMBRAL_TEJIDO
    tiene_tejido_2 = fila["fraccion_tejido_m2"] >= UMBRAL_TEJIDO
    if tiene_tejido_1 != tiene_tejido_2:
        return "sin_tejido_correspondiente"

    if tiene_tejido_1 and tiene_tejido_2 and fila["shift_mag_um"] >= p95_shift:
        return "mal_alineado_real"
    return "bien_alineado"


df_tiles_completo["categoria"] = df_tiles_completo.apply(clasificar, axis=1)
conteo = df_tiles_completo["categoria"].value_counts()

ruta_csv_v2 = os.path.join(CARPETA_TILES, "indice_tiles_con_metricas_v2.csv")
df_tiles_completo.to_csv(ruta_csv_v2, index=False)

N_ZOOM = 6
reales = (
    df_tiles_completo[df_tiles_completo["categoria"] == "mal_alineado_real"]
    .sort_values("shift_mag_um", ascending=False)
    .head(N_ZOOM)
)

if len(reales) > 0:
    fig, axes = plt.subplots(len(reales), 2, figsize=(7, len(reales) * 3.3))
    if len(reales) == 1:
        axes = [axes]
    for i, (_, fila) in enumerate(reales.iterrows()):
        tile1 = cv2.cvtColor(cv2.imread(os.path.join(carpeta_tiles_1, fila["nombre"])), cv2.COLOR_BGR2RGB)
        tile2 = cv2.cvtColor(cv2.imread(os.path.join(carpeta_tiles_2, fila["nombre"])), cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(tile1); axes[i][0].axis("off")
        axes[i][0].set_title(f"Muestra 1\nfila {int(fila['fila'])}, col {int(fila['col'])}", fontsize=9)
        axes[i][1].imshow(tile2); axes[i][1].axis("off")
        axes[i][1].set_title(
            f"Muestra 2\nshift={fila['shift_mag_um']:.0f}µm   resp={fila['phase_corr_response']:.2f}",
            fontsize=9
        )

    plt.suptitle("Peores tiles con tejido", y=1.0)
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_TILES, "zoom_mal_alineados_reales.png"), dpi=150, bbox_inches="tight")
    plt.show()

print(f"Umbral de falla de medición (percentil {PERCENTIL_FALLA_MEDICION:.1%}): "
      f"{UMBRAL_FALLA_MEDICION_UM:.0f} µm")
print(conteo)


In [ ]:
candidatos_crosscheck = df_tiles_completo[
    df_tiles_completo["categoria"] == "mal_alineado_real"
].copy()

resultados_ecc = []
for _, fila in candidatos_crosscheck.iterrows():
    tile1, tile2 = leer_par_tiles(fila["nombre"], carpeta_tiles_1, carpeta_tiles_2)
    if tile1 is None:
        continue

    m = metrica_ecc_tiles(
        tile1, tile2, mpp=mpp_base,
        shift_inicial=(fila["shift_x_px"], fila["shift_y_px"]),
    )
    m["nombre"] = fila["nombre"]
    resultados_ecc.append(m)

df_ecc = pd.DataFrame(resultados_ecc)
candidatos_crosscheck = candidatos_crosscheck.merge(df_ecc, on="nombre")

piso_confirmacion_um = df_tiles_completo["shift_mag_um"].median() * 3
candidatos_crosscheck["confirmado_por_ecc"] = (
    candidatos_crosscheck["shift_mag_um_ecc"] >= piso_confirmacion_um
)

n_confirmados = candidatos_crosscheck["confirmado_por_ecc"].sum()
n_total = len(candidatos_crosscheck)

ruta_csv_crosscheck = os.path.join(CARPETA_TILES, "indice_tiles_crosscheck_ecc.csv")
candidatos_crosscheck.to_csv(ruta_csv_crosscheck, index=False)

print(f"De {n_total} tiles marcados 'mal_alineado_real' por phase correlation, "
      f"ECC confirma {n_confirmados} ({n_confirmados / n_total * 100:.0f}%)")
candidatos_crosscheck[[
    "nombre", "fila", "col", "shift_mag_um", "phase_corr_response",
    "shift_mag_um_ecc", "ecc_score", "confirmado_por_ecc"
]].round(2)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))
paso_muestra = max(1, img_ref_completa.shape[1] // 2000)
ax.imshow(img_ref_completa[::paso_muestra, ::paso_muestra])
esc = 1.0 / paso_muestra

colores = {
    "mal_alineado_real": ("red", "o"),
    "sin_tejido_correspondiente": ("gray", "s"),
    "falla_medicion": ("deepskyblue", "x"),
}
for cat, (color, marker) in colores.items():
    sub = df_tiles_completo[df_tiles_completo["categoria"] == cat]
    cx = (sub["x1"] + sub["x2"]) / 2 * esc
    cy = (sub["y1"] + sub["y2"]) / 2 * esc
    ax.scatter(cx, cy, facecolors="none", edgecolors=color, marker=marker, s=60, linewidths=1.5, label=cat)

ax.legend(loc="upper right")
ax.set_title("Mapa de calidad de alineación — diferenciando error real vs. tejido ausente")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_TILES, "mapa_zonas_problematicas_v2.png"), dpi=150, bbox_inches="tight")
plt.show()


---
### Control visual — tiles bien alineados


In [ ]:
import os
import random
import cv2
import pandas as pd
import matplotlib.pyplot as plt

RUTA_CSV = os.path.join(CARPETA_TILES, "indice_tiles_con_metricas_v2.csv")
CARPETA_TILES_1 = os.path.join(CARPETA_TILES, "muestra_1")
CARPETA_TILES_2 = os.path.join(CARPETA_TILES, "muestra_2")

N_MUESTRA = 10

random.seed(SEMILLA)

df = pd.read_csv(RUTA_CSV)

df_bien = df[
    (df["categoria"] == "bien_alineado")
    & (df["shift_mag_um"] <= SHIFT_MAX_UM_CONTROL_VISUAL)
    & (df["phase_corr_response"] >= RESPONSE_MIN_CONTROL_VISUAL)
]

if len(df_bien) == 0:
    raise RuntimeError(
        "Ningún tile cumple categoria=='bien_alineado', "
        f"shift_mag_um<={SHIFT_MAX_UM_CONTROL_VISUAL} y phase_corr_response>={RESPONSE_MIN_CONTROL_VISUAL}."
    )

n = min(N_MUESTRA, len(df_bien))
muestra = df_bien.sample(n=n, random_state=SEMILLA)

fig, axes = plt.subplots(n, 2, figsize=(8, n * 4))
if n == 1:
    axes = [axes]

for i, (_, fila) in enumerate(muestra.iterrows()):
    nombre = fila["nombre"]

    ruta_1 = os.path.join(CARPETA_TILES_1, nombre)
    ruta_2 = os.path.join(CARPETA_TILES_2, nombre)

    tile1 = cv2.cvtColor(cv2.imread(ruta_1), cv2.COLOR_BGR2RGB)
    tile2 = cv2.cvtColor(cv2.imread(ruta_2), cv2.COLOR_BGR2RGB)

    axes[i][0].imshow(tile1)
    axes[i][0].set_title(f"Muestra 1\nfila {int(fila['fila'])}, col {int(fila['col'])}", fontsize=9)
    axes[i][0].axis("off")

    axes[i][1].imshow(tile2)
    axes[i][1].set_title(
        f"Muestra 2\nshift={fila['shift_mag_um']:.0f}µm   resp={fila['phase_corr_response']:.2f}",
        fontsize=9,
    )
    axes[i][1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_TILES, "muestra_tiles_bien_alineados.png"), dpi=150, bbox_inches="tight")
plt.show()


---
### Corrección fina de offset (opcional)


In [ ]:
import os
import random
import cv2
import numpy as np
import pandas as pd
import pyvips

N_TILES_FIX    = 8
ESCALA_GRUESA  = 0.1
MARGEN_FINO_PX = 150
MARGEN_BUSQ_X  = 10000
MARGEN_BUSQ_Y  = 4000

ruta_csv_tiles = os.path.join(CARPETA_TILES, "indice_tiles.csv")
df_fix = pd.read_csv(ruta_csv_tiles)

if "T1_VALIS" not in globals():
    T1_VALIS = os.path.join(CARPETA_TILES, "muestra_1")


def _vips_a_numpy(v):
    buf = v.write_to_memory()
    return np.ndarray(buffer=buf, dtype=np.uint8, shape=[v.height, v.width, v.bands])


img_tif = pyvips.Image.new_from_file(RUTA_TIF_ORIGINAL, access="random")
ancho_tif, alto_tif = img_tif.width, img_tif.height

off1 = offsets_recorte["muestra_1"] if "offsets_recorte" in globals() else \
       {"x": 3026, "y": 13997, "ancho": 17966, "alto": 19283}
bx0 = max(0, off1["x"] - MARGEN_BUSQ_X)
by0 = max(0, off1["y"] - MARGEN_BUSQ_Y)
bx1 = min(ancho_tif,  off1["x"] + off1["ancho"] + MARGEN_BUSQ_X)
by1 = min(alto_tif,   off1["y"] + off1["alto"]  + MARGEN_BUSQ_Y)

region_grande_vips = img_tif.crop(bx0, by0, bx1 - bx0, by1 - by0)
region_chica = _vips_a_numpy(region_grande_vips.resize(ESCALA_GRUESA))
region_chica_gray = cv2.cvtColor(region_chica[:, :, :3], cv2.COLOR_RGB2GRAY)

muestreo = df_fix.sample(min(N_TILES_FIX, len(df_fix)), random_state=SEMILLA)

resultados = []
for _, fila in muestreo.iterrows():
    tile = cv2.imread(os.path.join(T1_VALIS, fila["nombre"]))
    if tile is None:
        continue
    tile_gray = cv2.cvtColor(tile, cv2.COLOR_BGR2GRAY)
    th, tw = tile_gray.shape[:2]

    tile_chico = cv2.resize(tile_gray, (int(tw * ESCALA_GRUESA), int(th * ESCALA_GRUESA)))
    res_g = cv2.matchTemplate(region_chica_gray, tile_chico, cv2.TM_CCOEFF_NORMED)
    _, _, _, loc_g = cv2.minMaxLoc(res_g)
    x_ap, y_ap = int(loc_g[0] / ESCALA_GRUESA), int(loc_g[1] / ESCALA_GRUESA)

    fx0, fy0 = max(0, x_ap - MARGEN_FINO_PX), max(0, y_ap - MARGEN_FINO_PX)
    fx1 = min(bx1 - bx0, x_ap + tw + MARGEN_FINO_PX)
    fy1 = min(by1 - by0, y_ap + th + MARGEN_FINO_PX)
    parche = _vips_a_numpy(region_grande_vips.crop(fx0, fy0, fx1 - fx0, fy1 - fy0))
    parche_gray = cv2.cvtColor(parche[:, :, :3], cv2.COLOR_RGB2GRAY)

    res_f = cv2.matchTemplate(parche_gray, tile_gray, cv2.TM_CCOEFF_NORMED)
    val_f, loc_f = cv2.minMaxLoc(res_f)[1], cv2.minMaxLoc(res_f)[3]

    x1_real = bx0 + fx0 + loc_f[0]
    y1_real = by0 + fy0 + loc_f[1]
    dx, dy = x1_real - fila["x1_orig"], y1_real - fila["y1_orig"]
    resultados.append({"tile": fila["nombre"], "dx": dx, "dy": dy, "corr": val_f})

df_res = pd.DataFrame(resultados)
dx_med, dy_med = df_res["dx"].median(), df_res["dy"].median()

if df_res["corr"].mean() > 0.5 and df_res["dx"].std() < 25 and df_res["dy"].std() < 25:
    df_fix["x1_orig"] += dx_med
    df_fix["y1_orig"] += dy_med
    df_fix.to_csv(ruta_csv_tiles, index=False)
    _aplicado = True
else:
    _aplicado = False

print(f"Offset medido: dx={dx_med:.0f}, dy={dy_med:.0f} "
      f"(std dx={df_res.dx.std():.1f}, dy={df_res.dy.std():.1f}, corr media={df_res['corr'].mean():.2f})")
print("Corrección aplicada a indice_tiles.csv." if _aplicado
      else "Offset no consistente entre tiles: no se aplicó corrección automática.")


---
## Paso 7 — Ubicación de tiles problemáticos en ambas imágenes


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

if "CARPETA_TILES" not in globals():
    CARPETA_TILES = f"{CARPETA_TILES_BASE}_tile{TILE_SIZE}_nivel{NIVEL_TILES}"

if "carpeta_tiles_1" not in globals() or "carpeta_tiles_2" not in globals():
    carpeta_tiles_1 = os.path.join(CARPETA_TILES, "muestra_1")
    carpeta_tiles_2 = os.path.join(CARPETA_TILES, "muestra_2")

if "df_tiles_completo" not in globals():
    _ruta_v2 = os.path.join(CARPETA_TILES, "indice_tiles_con_metricas_v2.csv")
    _ruta_v1 = os.path.join(CARPETA_TILES, "indice_tiles_con_metricas.csv")
    if os.path.exists(_ruta_v2):
        df_tiles_completo = pd.read_csv(_ruta_v2)
    elif os.path.exists(_ruta_v1):
        # Sin columna 'categoria'; se usa el criterio de percentil como respaldo.
        df_tiles_completo = pd.read_csv(_ruta_v1)
    else:
        raise RuntimeError(f"No encontré {_ruta_v2} ni {_ruta_v1}. Corré el Paso 4 (y el Paso 6) primero.")

MINI = 20
N_MARCAR = 12

n_filas_grid = df_tiles_completo["fila"].max() + 1
n_cols_grid  = df_tiles_completo["col"].max() + 1


def construir_mosaico(carpeta_tiles):
    mosaico = np.full((n_filas_grid * MINI, n_cols_grid * MINI, 3), 255, dtype=np.uint8)
    for _, fila in df_tiles_completo.iterrows():
        ruta_tile = os.path.join(carpeta_tiles, fila["nombre"])
        if not os.path.exists(ruta_tile):
            continue
        tile_chico = np.array(Image.open(ruta_tile).convert("RGB").resize((MINI, MINI)))
        f, c = int(fila["fila"]), int(fila["col"])
        mosaico[f*MINI:(f+1)*MINI, c*MINI:(c+1)*MINI] = tile_chico
    return mosaico


mosaico_1 = construir_mosaico(carpeta_tiles_1)
mosaico_2 = construir_mosaico(carpeta_tiles_2)

if "categoria" in df_tiles_completo.columns:
    candidatos = df_tiles_completo[df_tiles_completo["categoria"] == "mal_alineado_real"]
else:
    UMBRAL = df_tiles_completo["shift_mag_um"].quantile(0.95)
    candidatos = df_tiles_completo[df_tiles_completo["shift_mag_um"] >= UMBRAL]

a_marcar = candidatos.sort_values("shift_mag_um", ascending=False).head(N_MARCAR).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(20, max(6, 10 * n_filas_grid / n_cols_grid)))

for ax, mosaico, titulo in zip(axes, [mosaico_1, mosaico_2], ["Muestra 1 (referencia)", "Muestra 2 (alineada)"]):
    ax.imshow(mosaico)
    ax.set_title(titulo)
    ax.axis("off")

for i, fila in a_marcar.iterrows():
    f, c = int(fila["fila"]), int(fila["col"])
    cx, cy = c * MINI + MINI / 2, f * MINI + MINI / 2
    for ax in axes:
        ax.scatter([cx], [cy], s=140, facecolors="none", edgecolors="red", linewidths=1.8)
        ax.annotate(str(i + 1), (cx, cy), color="red", fontsize=9, weight="bold",
                    ha="center", va="center")

plt.suptitle("Mismos tiles marcados en las dos imágenes (mismo grid fila/col)", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_TILES, "ubicacion_tiles_ambas_imagenes.png"), dpi=150, bbox_inches="tight")
plt.show()

tabla = a_marcar[["fila", "col", "nombre", "shift_mag_um", "phase_corr_response"]].copy()
tabla.insert(0, "n°", range(1, len(tabla) + 1))
tabla.round(1)


---
## Paso 8 — Comparación de sectores vs. TIF original (VALIS vs. GrandQC)


In [ ]:
import os
import re
import sys
import shutil
import numpy as np
import pandas as pd
import cv2
import pyvips
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_RUTA_PIPELINE_LIB = r"C:\investigacion\repo_alineacion_kidney\repo"
if _RUTA_PIPELINE_LIB not in sys.path:
    sys.path.insert(0, _RUTA_PIPELINE_LIB)

from pipeline_lib.sector_comparison import ComparadorSectores, cargar_indice_gqc, cargar_indice_valis

T1_VALIS  = os.path.join(CARPETA_TILES, "muestra_1")
T2_VALIS  = os.path.join(CARPETA_TILES, "muestra_2")
CSV_VALIS = os.path.join(CARPETA_TILES, "indice_tiles.csv")
TS_VALIS  = TILE_SIZE


def _enlazar_o_copiar(origen, destino):
    try:
        os.link(origen, destino)
    except OSError:
        shutil.copy2(origen, destino)


def _construir_carpeta_combinada_gqc(base_gqc, subcarpeta_tejido, nombre_carpeta_combinada="rescate_ecc_geom"):
    # Combina los tiles de GrandQC alineados por ECC con los alineados por
    # geometría (usados como rescate donde ECC no convergió).
    carpeta_ecc = os.path.join(base_gqc, "ecc", subcarpeta_tejido)
    carpeta_geom = os.path.join(base_gqc, "geom", subcarpeta_tejido)
    carpeta_combinada = os.path.join(base_gqc, nombre_carpeta_combinada, subcarpeta_tejido)
    os.makedirs(carpeta_combinada, exist_ok=True)

    n_ecc, n_rescatados = 0, 0

    if os.path.isdir(carpeta_ecc):
        for nombre in os.listdir(carpeta_ecc):
            destino = os.path.join(carpeta_combinada, nombre)
            if not os.path.exists(destino):
                _enlazar_o_copiar(os.path.join(carpeta_ecc, nombre), destino)
            n_ecc += 1

    if os.path.isdir(carpeta_geom):
        for nombre in os.listdir(carpeta_geom):
            destino = os.path.join(carpeta_combinada, nombre)
            if not os.path.exists(destino):
                _enlazar_o_copiar(os.path.join(carpeta_geom, nombre), destino)
                n_rescatados += 1

    print(f"{subcarpeta_tejido}: {n_ecc} tiles de ecc/ + {n_rescatados} rescatados de geom/ "
          f"= {n_ecc + n_rescatados} tiles -> {carpeta_combinada}")
    return carpeta_combinada


BASE_GQC = "C:/investigacion/datasetNefroTileCALIDAD"
CSV_GQC  = os.path.join(BASE_GQC, "manifest.csv")
TS_GQC   = 512

USAR_RESCATE_GEOM = True

if USAR_RESCATE_GEOM:
    T1_GQC = _construir_carpeta_combinada_gqc(BASE_GQC, "tejido1")
    T2_GQC = _construir_carpeta_combinada_gqc(BASE_GQC, "tejido2")
else:
    T1_GQC = os.path.join(BASE_GQC, "ecc", "tejido1")
    T2_GQC = os.path.join(BASE_GQC, "ecc", "tejido2")

SALIDA = os.path.join(CARPETA_RESULTADOS, "comparacion_pipelines")
os.makedirs(SALIDA, exist_ok=True)

N_SECTORES = 20

VENTANA_PX = 1536
COLS = 4

MIN_COB_REF = 0.50
MIN_COB_V = 0.50
MIN_COB_G = 0.30

RELLENO = 220
UMBRAL_BLANCO = 230
UMBRAL_TEXTURA = 15

MAX_HUECO_G = 0.15
FACTOR_SOBREMUESTREO = 3

SZ_PANEL = 220
DPI_MOSAICO = 100

CV = "#1a4f8a"
CG = "#8a2a1a"
CR = "#1a6b1a"

comparador = ComparadorSectores(
    relleno=RELLENO, umbral_blanco=UMBRAL_BLANCO, umbral_textura=UMBRAL_TEXTURA, semilla=SEMILLA,
)


def hueco_maximo_contiguo(canvas, relleno=None, tol=2):
    if relleno is None:
        relleno = RELLENO
    gris = np.all(np.abs(canvas.astype(int) - relleno) <= tol, axis=-1).astype(np.uint8)
    n_labels, _, stats, _ = cv2.connectedComponentsWithStats(gris, connectivity=8)
    if n_labels <= 1:
        return 0
    return int(stats[1:, cv2.CC_STAT_AREA].max())


_PATRON_NOMBRE_GQC = re.compile(r"^tile_\d+_r(?P<y1>\d+)_c(?P<x1>\d+)\.\w+$", re.IGNORECASE)

df_gqc = cargar_indice_gqc(CSV_GQC, T1_GQC, _PATRON_NOMBRE_GQC)
df_valis = cargar_indice_valis(CSV_VALIS)

if not os.path.exists(RUTA_TIF_ORIGINAL):
    raise RuntimeError(f"No se encontró el TIF original en {RUTA_TIF_ORIGINAL}.")
tif_vips = pyvips.Image.new_from_file(RUTA_TIF_ORIGINAL, access="random")

# Se piden más candidatos de los necesarios (FACTOR_SOBREMUESTREO) para
# poder filtrar por hueco máximo contiguo (MAX_HUECO_G) y quedarse con
# N_SECTORES sectores con cobertura real verificada contra el TIF original.
candidatos_sectores = comparador.elegir_sectores(
    N_SECTORES * FACTOR_SOBREMUESTREO, VENTANA_PX, tif_vips,
    df_valis, T1_VALIS, T2_VALIS, TS_VALIS,
    df_gqc, T1_GQC, T2_GQC, TS_GQC,
    MIN_COB_REF, MIN_COB_V, MIN_COB_G,
)

sectores = []
descartados_por_hueco = 0
for s in candidatos_sectores:
    g2, _ = comparador.reconstruir_desde_tiles(df_gqc, T2_GQC, s["cx"], s["cy"], VENTANA_PX, TS_GQC)
    area_hueco = hueco_maximo_contiguo(g2)
    if area_hueco <= MAX_HUECO_G * (VENTANA_PX ** 2):
        sectores.append(s)
    else:
        descartados_por_hueco += 1
    if len(sectores) >= N_SECTORES:
        break


def thumb(arr, sz=SZ_PANEL):
    return cv2.resize(arr, (sz, sz), interpolation=cv2.INTER_AREA)


if sectores:
    FPS = 3
    n = len(sectores)
    bloqs = int(np.ceil(n / COLS))

    fig, axes = plt.subplots(
        bloqs * FPS, COLS,
        figsize=(COLS * SZ_PANEL / DPI_MOSAICO, bloqs * FPS * SZ_PANEL / DPI_MOSAICO),
        dpi=DPI_MOSAICO,
        squeeze=False,
    )
    for ax in axes.flat:
        ax.axis("off")

    for idx, s in enumerate(sectores):
        c, blq = idx % COLS, idx // COLS
        cx, cy = s["cx"], s["cy"]

        ref, _ = comparador.leer_referencia_real(tif_vips, cx, cy, VENTANA_PX)
        v2, _ = comparador.reconstruir_desde_tiles(df_valis, T2_VALIS, cx, cy, VENTANA_PX, TS_VALIS)
        g2, _ = comparador.reconstruir_desde_tiles(df_gqc, T2_GQC, cx, cy, VENTANA_PX, TS_GQC)

        paneles = [
            (blq * FPS + 0, ref, f"Referencia real (TIF)\n({cx},{cy})  cob={s['cob_ref']*100:.0f}%", CR),
            (blq * FPS + 1, v2, f"VALIS T2 alineado\ncorr vs ref={s['corr_v2']:.2f}", CV),
            (blq * FPS + 2, g2, f"GQC T2 alineado\ncorr vs ref={s['corr_g2']:.2f}", CG),
        ]
        for frow, img, titulo, col_txt in paneles:
            ax = axes[frow][c]
            ax.imshow(thumb(img))
            ax.axis("off")
            ax.set_title(titulo, fontsize=5.5, pad=1, color=col_txt)

    for blq in range(bloqs):
        for rel, lbl, col_txt in [(0, "Referencia real", CR), (1, "VALIS T2", CV), (2, "GQC T2", CG)]:
            axes[blq * FPS + rel][0].set_ylabel(lbl, fontsize=7, rotation=90, labelpad=2, color=col_txt)

    plt.suptitle(
        f"Mismo sector, verificado contra el TIF original  ({n} sectores, ventana {VENTANA_PX}px)\n"
        "Fila 1: referencia real (TIF, sin pasar por ningún pipeline)   "
        "Filas 2-3: alineado de VALIS / GrandQC, comparado contra esa referencia",
        fontsize=8, y=1.005,
    )
    plt.tight_layout(pad=0.25)
    ruta_mos = os.path.join(SALIDA, f"mosaico_{n}sectores.png")
    plt.savefig(ruta_mos, dpi=DPI_MOSAICO, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def figura_detalle(s, vpx=VENTANA_PX):
    cx, cy = s["cx"], s["cy"]
    ref, _ = comparador.leer_referencia_real(tif_vips, cx, cy, vpx)
    v2, _ = comparador.reconstruir_desde_tiles(df_valis, T2_VALIS, cx, cy, vpx, TS_VALIS)
    g2, _ = comparador.reconstruir_desde_tiles(df_gqc, T2_GQC, cx, cy, vpx, TS_GQC)
    n_v2 = comparador.contar_tiles_en_ventana(df_valis, cx, cy, vpx, TS_VALIS)
    n_g2 = comparador.contar_tiles_en_ventana(df_gqc, cx, cy, vpx, TS_GQC)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
    for ax, img, titulo, col_txt in [
        (axes[0], ref, f"Referencia real (TIF original)\ncob={s['cob_ref']*100:.0f}%", CR),
        (axes[1], v2, f"VALIS -- T2 alineado ({n_v2} tiles)\ncorr vs referencia = {s['corr_v2']:.2f}", CV),
        (axes[2], g2, f"GrandQC -- T2 alineado ({n_g2} tiles)\ncorr vs referencia = {s['corr_g2']:.2f}", CG),
    ]:
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(titulo, fontsize=11, color=col_txt, pad=4)

    plt.suptitle(
        f"Sector ({cx}, {cy}) -- ventana {vpx}px    "
        f"[sanidad T1: VALIS={s['corr_v1_sanidad']:.2f}, GQC={s['corr_g1_sanidad']:.2f}]",
        fontsize=12,
    )
    plt.tight_layout()
    ruta = os.path.join(SALIDA, f"detalle_{cx}_{cy}.png")
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for s in sectores[:4]:
    figura_detalle(s)

print(f"Rango VALIS   x1_orig: {df_valis.x1_orig.min()}-{df_valis.x1_orig.max()}, "
      f"y1_orig: {df_valis.y1_orig.min()}-{df_valis.y1_orig.max()}")
print(f"Rango GrandQC x1_orig: {df_gqc.x1_orig.min()}-{df_gqc.x1_orig.max()}, "
      f"y1_orig: {df_gqc.y1_orig.min()}-{df_gqc.y1_orig.max()}")
print(f"Sectores: {len(sectores)}/{N_SECTORES} pasaron el filtro de hueco máximo "
      f"({descartados_por_hueco} descartados, de {len(candidatos_sectores)} candidatos evaluados)")
print(f"Resultados guardados en: {SALIDA}")


---
## Paso 9 — VALIS vs. tiatoolbox


In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

_carpetas_motor = {
    "valis":      f"{CARPETA_TILES_BASE}_tile{TILE_SIZE}_nivel{NIVEL_TILES}_valis",
    "tiatoolbox": f"{CARPETA_TILES_BASE}_tile{TILE_SIZE}_nivel{NIVEL_TILES}_tiatoolbox",
}


def _cargar_resultados_motor(motor, carpeta):
    ruta_v2 = os.path.join(carpeta, "indice_tiles_con_metricas_v2.csv")
    ruta_v1 = os.path.join(carpeta, "indice_tiles_con_metricas.csv")
    if os.path.exists(ruta_v2):
        df = pd.read_csv(ruta_v2)
    elif os.path.exists(ruta_v1):
        df = pd.read_csv(ruta_v1)  # sin columna 'categoria'
    else:
        return None, None

    ruta_tiempo = os.path.join(CARPETA_RESULTADOS, f"tiempo_registro_{motor}.json")
    tiempos = None
    if os.path.exists(ruta_tiempo):
        with open(ruta_tiempo) as _f:
            tiempos = json.load(_f)
    return df, tiempos


filas_resumen = []
dfs_por_motor = {}
for motor, carpeta in _carpetas_motor.items():
    df_m, tiempos_m = _cargar_resultados_motor(motor, carpeta)
    if df_m is None:
        continue
    dfs_por_motor[motor] = df_m

    fila = {
        "motor": motor,
        "n_tiles": len(df_m),
        "mediana_shift_um": df_m["shift_mag_um"].median(),
        "media_shift_um": df_m["shift_mag_um"].mean(),
        "p95_shift_um": df_m["shift_mag_um"].quantile(0.95),
        "media_phase_corr_response": df_m["phase_corr_response"].mean(),
    }
    if "categoria" in df_m.columns:
        conteo = df_m["categoria"].value_counts(normalize=True) * 100
        fila["pct_mal_alineado_real"] = conteo.get("mal_alineado_real", 0.0)
        fila["pct_sin_tejido_correspondiente"] = conteo.get("sin_tejido_correspondiente", 0.0)
        fila["pct_falla_medicion"] = conteo.get("falla_medicion", 0.0)
    if tiempos_m is not None:
        fila["segundos_registro_total"] = tiempos_m.get("segundos_total")
    filas_resumen.append(fila)

if len(filas_resumen) < 2:
    df_resumen = pd.DataFrame(filas_resumen).round(2)
else:
    df_resumen = pd.DataFrame(filas_resumen).set_index("motor")

    ruta_resumen = os.path.join(CARPETA_RESULTADOS, "comparacion_valis_vs_tiatoolbox.csv")
    df_resumen.to_csv(ruta_resumen)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    df_resumen["mediana_shift_um"].plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"])
    axes[0].set_title("Desplazamiento residual\nmediana (µm) — menor es mejor")
    axes[0].set_xlabel("")

    df_resumen["media_phase_corr_response"].plot.bar(ax=axes[1], color=["#4C72B0", "#DD8452"])
    axes[1].set_title("Confianza de la métrica\n(phase_corr_response) — mayor es mejor")
    axes[1].set_xlabel("")

    if "segundos_registro_total" in df_resumen.columns:
        (df_resumen["segundos_registro_total"] / 60).plot.bar(ax=axes[2], color=["#4C72B0", "#DD8452"])
        axes[2].set_title("Tiempo de registro (min)")
    else:
        axes[2].axis("off")
    axes[2].set_xlabel("")

    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_RESULTADOS, "comparacion_valis_vs_tiatoolbox.png"), dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot(
        [dfs_por_motor[m]["shift_mag_um"].dropna() for m in dfs_por_motor],
        tick_labels=list(dfs_por_motor.keys()),
        showfliers=False,
    )
    ax.set_ylabel("Desplazamiento residual (µm)")
    ax.set_title("Distribución de shift_mag_um por motor")
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_RESULTADOS, "boxplot_valis_vs_tiatoolbox.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Menor desplazamiento residual (mediana): {df_resumen['mediana_shift_um'].idxmin()}")
    print(f"Mayor confianza de la métrica (promedio): {df_resumen['media_phase_corr_response'].idxmax()}")
    if "pct_mal_alineado_real" in df_resumen.columns:
        print(f"Menor porcentaje de tiles 'mal_alineado_real': {df_resumen['pct_mal_alineado_real'].idxmin()}")

df_resumen


---
## Paso 10 — Exportación de parámetros


In [ ]:
import os
import json

if "mpp_base" not in globals() or mpp_base is None:
    with open(os.path.join(CARPETA_RESULTADOS, "metadata_pipeline.json")) as _f:
        mpp_base = json.load(_f)["mpp_base"]

parametros = {
    "motor_registro_usado_en_esta_corrida": MOTOR_REGISTRO,
    "mpp_base_um_por_px": mpp_base,
    "grilla_de_tiles": {
        "tile_size_px": TILE_SIZE,
        "stride_px": STRIDE,
        "nivel_tiles": NIVEL_TILES,
        "tile_size_um": TILE_SIZE * mpp_base,
        "stride_um": STRIDE * mpp_base,
    },
    "umbrales_de_descarte_de_tiles": {
        "umbral_tejido_fraccion_minima": UMBRAL_TEJIDO,
        "min_informatividad_var_laplaciano": MIN_INFORMATIVIDAD,
        "criterio_pixel_tejido": "gris < 240 (0-255) = tejido; >= 240 = fondo blanco",
    },
    "umbrales_de_clasificacion_de_calidad": {
        "umbral_response_falla_medicion": UMBRAL_RESPONSE_FALLA,
        "umbral_shift_falla_relativo_x_mediana": UMBRAL_SHIFT_FALLA_RELATIVO,
        "percentil_falla_medicion": PERCENTIL_FALLA_MEDICION,
        "percentil_corte_mal_alineado_dentro_de_tiles_limpios": 0.95,
        "umbral_std_textura_chequeo_global": UMBRAL_STD_TEXTURA,
    },
    "umbrales_control_visual_bien_alineado": {
        "shift_max_um": SHIFT_MAX_UM_CONTROL_VISUAL,
        "response_min": RESPONSE_MIN_CONTROL_VISUAL,
    },
    "separacion_de_muestras": {
        "metodo": "separar_muestras.py (Otsu, determinista, mismo criterio en las 2 muestras del par izquierda->derecha)",
        "lado_miniatura_px": SEPARAR_LADO_MINIATURA_PX,
        "margen_pct": SEPARAR_MARGEN_PCT,
        "tamanio_min_objeto_px": SEPARAR_TAMANIO_MIN_OBJETO_PX,
    },
    "semilla_aleatoria": SEMILLA,
}

if MOTOR_REGISTRO == "valis":
    parametros["parametros_valis"] = {
        "max_processed_image_dim_px": VALIS_MAX_PROCESSED_IMG_DIM_PX,
        "max_non_rigid_registration_dim_px": VALIS_MAX_NON_RIGID_DIM_PX,
        "micro_registro_activado": CORRER_MICRO_REGISTRO,
        "micro_max_non_rigid_registration_dim_px": VALIS_MICRO_MAX_NON_RIGID_DIM_PX if CORRER_MICRO_REGISTRO else None,
        "features_profundos_disk_lightglue": VALIS_USAR_FEATURES_DEEP,
        "n_features": VALIS_N_FEATURES if VALIS_USAR_FEATURES_DEEP else None,
        "micro_rigid_registrar": VALIS_USAR_MICRO_RIGID_REGISTRAR,
        "check_for_reflections": VALIS_CHECK_FOR_REFLECTIONS,
        "non_rigid_registrar": VALIS_NON_RIGID_REGISTRAR,
    }
elif MOTOR_REGISTRO == "tiatoolbox":
    parametros["parametros_tiatoolbox"] = {
        "dfbr_max_dim_px_miniatura": TIATOOLBOX_DFBR_MAX_DIM_PX,
        "dispositivo": TIATOOLBOX_DFBR_DEVICE,
        "refinamiento_bspline_por_tile": TIATOOLBOX_REFINAR_BSPLINE_POR_TILE,
        "bspline_grid_space_um": TIATOOLBOX_BSPLINE_GRID_SPACE_UM if TIATOOLBOX_REFINAR_BSPLINE_POR_TILE else None,
        "bspline_sampling_percent": TIATOOLBOX_BSPLINE_SAMPLING_PERCENT if TIATOOLBOX_REFINAR_BSPLINE_POR_TILE else None,
    }

ruta_json = os.path.join(CARPETA_RESULTADOS, f"parametros_pipeline_alineacion_{MOTOR_REGISTRO}.json")
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(parametros, f, indent=2, ensure_ascii=False)

lineas = []
lineas.append(f"# Parámetros del pipeline de alineación — motor: {MOTOR_REGISTRO}")
lineas.append("")
lineas.append("Estos valores fijan la configuración usada en esta corrida y deben mantenerse")
lineas.append("constantes al comparar contra otros pipelines o motores de registro, de forma")
lineas.append("que las diferencias observadas no respondan a diferencias de configuración")
lineas.append("(tamaño de tile, mpp, umbrales de descarte de tejido, etc.).")
lineas.append("")
lineas.append("El mismo sistema de referencia (recorte de las 2 muestras) se reproduce a partir de")
lineas.append("`separar_muestras.py`, o de `muestra_1.tif` / `muestra_2.tif` / `offsets_recorte.json`.")
lineas.append("")
lineas.append("## Resolución")
lineas.append(f"- **mpp (µm/px) a resolución completa**: {mpp_base:.4f}")
lineas.append(f"- Nivel de pirámide usado (`NIVEL_TILES`): {NIVEL_TILES}")
lineas.append("")
lineas.append("## Separación de las 2 muestras")
lineas.append("Método: `separar_muestras.py` (Otsu + 2 regiones más grandes, izquierda a derecha, determinista).")
lineas.append(f"- `SEPARAR_LADO_MINIATURA_PX`: {SEPARAR_LADO_MINIATURA_PX}")
lineas.append(f"- `SEPARAR_MARGEN_PCT`: {SEPARAR_MARGEN_PCT}")
lineas.append(f"- `SEPARAR_TAMANIO_MIN_OBJETO_PX`: {SEPARAR_TAMANIO_MIN_OBJETO_PX}")
lineas.append("")
lineas.append("## Grilla de tiles")
lineas.append(f"- `TILE_SIZE`: {TILE_SIZE} px ({TILE_SIZE * mpp_base:.1f} µm de lado)")
lineas.append(f"- `STRIDE`: {STRIDE} px ({STRIDE * mpp_base:.1f} µm)")
lineas.append("")
lineas.append("## Umbrales de descarte de tiles")
lineas.append(f"- Fracción mínima de tejido por tile (`UMBRAL_TEJIDO`): {UMBRAL_TEJIDO} (= {UMBRAL_TEJIDO * 100:.0f}%)")
lineas.append(f"- Informatividad mínima — varianza del Laplaciano (`MIN_INFORMATIVIDAD`): {MIN_INFORMATIVIDAD}")
lineas.append("- Criterio de 'tejido' a nivel de píxel: gris < 240 (escala 0-255) = tejido; >= 240 = fondo blanco")
lineas.append("")
lineas.append("## Umbrales de clasificación de calidad")
lineas.append(f"- `UMBRAL_RESPONSE_FALLA`: {UMBRAL_RESPONSE_FALLA}")
lineas.append(f"- `UMBRAL_SHIFT_FALLA_RELATIVO`: {UMBRAL_SHIFT_FALLA_RELATIVO}x la mediana global de shift")
lineas.append(f"- `PERCENTIL_FALLA_MEDICION`: {PERCENTIL_FALLA_MEDICION:.1%}")
lineas.append("- Percentil usado como corte de 'mal alineado' dentro de los tiles limpios: 95")
lineas.append("")
lineas.append("## Semilla aleatoria")
lineas.append(f"- `SEMILLA`: {SEMILLA}")
lineas.append("")

if MOTOR_REGISTRO == "valis":
    lineas.append("## Parámetros específicos de VALIS")
    lineas.append(f"- `max_processed_image_dim_px`: {VALIS_MAX_PROCESSED_IMG_DIM_PX}")
    lineas.append(f"- `max_non_rigid_registration_dim_px`: {VALIS_MAX_NON_RIGID_DIM_PX}")
    _micro_txt = "activado" if CORRER_MICRO_REGISTRO else "desactivado"
    if CORRER_MICRO_REGISTRO:
        _micro_txt += f" (max_non_rigid_registration_dim_px={VALIS_MICRO_MAX_NON_RIGID_DIM_PX})"
    lineas.append(f"- Micro-registro (`register_micro`): {_micro_txt}")
    _feat_txt = f"DISK + LightGlue (n={VALIS_N_FEATURES})" if VALIS_USAR_FEATURES_DEEP else "detector por defecto de VALIS"
    lineas.append(f"- Features: {_feat_txt}")
    lineas.append(f"- MicroRigidRegistrar (refinamiento rígido extra): {'sí' if VALIS_USAR_MICRO_RIGID_REGISTRAR else 'no'}")
    lineas.append(f"- check_for_reflections: {VALIS_CHECK_FOR_REFLECTIONS}")
    lineas.append(f"- Registrador no-rígido: {VALIS_NON_RIGID_REGISTRAR}")
elif MOTOR_REGISTRO == "tiatoolbox":
    lineas.append("## Parámetros específicos de tiatoolbox")
    lineas.append(f"- Tamaño de miniatura para DFBR: ~{TIATOOLBOX_DFBR_MAX_DIM_PX} px de lado mayor")
    _bspline_txt = "activado" if TIATOOLBOX_REFINAR_BSPLINE_POR_TILE else "desactivado"
    if TIATOOLBOX_REFINAR_BSPLINE_POR_TILE:
        _bspline_txt += f" (grid_space={TIATOOLBOX_BSPLINE_GRID_SPACE_UM} µm, sampling_percent={TIATOOLBOX_BSPLINE_SAMPLING_PERCENT})"
    lineas.append(f"- Refinamiento B-spline por tile: {_bspline_txt}")

lineas.append("")
lineas.append("## Métricas de calidad")
lineas.append("- **shift_mag_px / shift_mag_um**: magnitud del desplazamiento residual estimado por")
lineas.append("  *phase correlation* sobre el mapa de gradiente (Sobel + CLAHE) de cada par de tiles")
lineas.append("  (compara estructura, no color, por lo que es robusto a diferencias de tinción entre muestras).")
lineas.append("- **phase_corr_response**: confianza del pico de correlación de fase (0 a 1).")
lineas.append("- **ecc_score / shift_mag_um_ecc**: verificación cruzada con ECC (Enhanced Correlation")
lineas.append("  Coefficient, MOTION_EUCLIDEAN), tolerante a diferencias de brillo/contraste entre tinciones.")

md_texto = "\n".join(lineas)
ruta_md = os.path.join(CARPETA_RESULTADOS, f"parametros_pipeline_alineacion_{MOTOR_REGISTRO}.md")
with open(ruta_md, "w", encoding="utf-8") as f:
    f.write(md_texto)

print(f"Guardado: {ruta_json}")
print(f"Guardado: {ruta_md}")
